# Dataset Investigation

## Objective
Investigate the datasets used by our existing speech-emotion recognition model before designing a new fine-tuning experiment.

## Datasets
- TESS
- CREMA-D
- SAVEE
- RAVDESS

## Questions
1. How many recordings and speakers does each dataset contain?
2. Which emotion labels are available?
3. How were the recordings collected?
4. How should emotion labels be mapped consistently?
5. Can we create a speaker-independent evaluation split?

## Findings from the Original Model Card

**Base checkpoint:** `facebook/wav2vec2-base-960h`

**Emotion model:** `Dpngtm/wav2vec2-emotion-recognition`

**Reported training datasets:**
- TESS
- CREMA-D
- SAVEE
- RAVDESS

**Reported training approach:**
- Frozen CNN feature extractor
- Trainable sequence-classification head
- 10 epochs
- Learning rate: 3e-5
- Cosine learning-rate scheduler
- Physical batch size: 4
- Effective batch size: 16 using gradient accumulation

**Reported performance:**
- Accuracy: 79.94%
- F1 score: 79.65%

**Reported label convention:** Calm and neutral are merged into a single neutral class.

**Information still to verify:**
- Exact number of recordings used from each dataset
- Dataset-specific preprocessing and label mapping
- Speaker-independent train/validation/test split
- Whether the reported metrics can be reproduced

**Research objective:** Investigate whether a carefully controlled, multi-dataset fine-tuning experiment can improve generalization to unseen speakers.

| Dataset | Recordings | Speakers | Emotion classes |
|---|---:|---:|---|
| CREMA-D | 7,442 | 91 | Angry, disgust, fear, happy, neutral, sad |
| RAVDESS | 1,440 unique recordings | 24 | Angry, calm, disgust, fearful, happy, neutral, sad, surprised |
| TESS | 2,800 | 2 | Angry, disgust, fear, happy, neutral, pleasant surprise, sad |
| SAVEE | 480 | 4 | Angry, disgust, fear, happy, neutral, sad, surprise |

### Step 2: Standardizing Emotion Labels

We will use a common set of eight target emotion classes:

`angry`, `calm`, `disgust`, `fearful`, `happy`, `neutral`, `sad`, `surprised`

Label mapping rules:
- `anger` → `angry`
- `fear` → `fearful`
- `happiness` → `happy`
- `pleasant surprise` or `surprise` → `surprised`
- `calm` and `neutral` will remain separate classes.

We will not assign labels that are absent from a dataset. Each recording will retain its original emotion label, converted only when an equivalent target class exists.

In [ ]:

from pathlib import Path

project_dir = Path("/content/drive/MyDrive/SpeechEmotionProject")

for name in ["ravdess.zip", "CREMA-D", "TESS", "SAVEE"]:
    path = project_dir / name
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")


ravdess.zip: NOT FOUND
CREMA-D: NOT FOUND
TESS: NOT FOUND
SAVEE: NOT FOUND


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

print("Folders and files containing 'emotion', 'ravdess', 'crema', 'tess', or 'savee':")

keywords = ["emotion", "ravdess", "crema", "tess", "savee"]

for path in drive_root.rglob("*"):
    if any(word in path.name.lower() for word in keywords):
        print(path)


Mounted at /content/drive
Folders and files containing 'emotion', 'ravdess', 'crema', 'tess', or 'savee':
/content/drive/MyDrive/SpeechEmotionProject
/content/drive/MyDrive/LinkedIn Certificates/CertificateOfCompletion_Emotional Intelligence Basics.pdf
/content/drive/MyDrive/Colab Notebooks/speech_emotion_finetuning_v3.ipynb
/content/drive/MyDrive/Speech_emo/RAVDESS
/content/drive/MyDrive/SpeechEmotionProject/ravdess.zip


In [ ]:

from pathlib import Path

paths = [
    Path("/content/drive/MyDrive/SpeechEmotionProject"),
    Path("/content/drive/MyDrive/Speech_emo/RAVDESS"),
]

for folder in paths:
    print(f"\n--- {folder} ---")
    if folder.exists():
        for item in list(folder.iterdir())[:15]:
            print(item.name, "(folder)" if item.is_dir() else "(file)")
    else:
        print("Folder not found")



--- /content/drive/MyDrive/SpeechEmotionProject ---
ravdess.zip (file)
checkpoints (folder)
final_model (folder)
experiment_2 (folder)
final_model_v2 (folder)

--- /content/drive/MyDrive/Speech_emo/RAVDESS ---


In [ ]:

import zipfile

zip_path = "/content/drive/MyDrive/SpeechEmotionProject/ravdess.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files = [f for f in z.namelist() if f.lower().endswith(".wav")]
    print("Total WAV files in ZIP:", len(files))
    print("Example file paths:")
    for f in files[:5]:
        print(f)


Total WAV files in ZIP: 2880
Example file paths:
Actor_01/03-01-01-01-01-01-01.wav
Actor_01/03-01-01-01-01-02-01.wav
Actor_01/03-01-01-01-02-01-01.wav
Actor_01/03-01-01-01-02-02-01.wav
Actor_01/03-01-02-01-01-01-01.wav


In [ ]:

!git lfs version


git: 'lfs' is not a git command. See 'git --help'.

The most similar command is
	log


In [ ]:

!apt-get update -qq && apt-get install -y git-lfs
!git lfs version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 53 not upgraded.
Need to get 3,908 kB of archives.
After this operation, 11.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 git-lfs amd64 3.4.1-1ubuntu0.4 [3,908 kB]
Fetched 3,908 kB in 2s (2,196 kB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...
git-lfs/3.4.1 (GitHub; li

In [ ]:

!df -h /content


Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  20% /


In [ ]:

!git lfs install
!git clone https://gitlab.com/cs-cooper-lab/crema-d-mirror.git /content/CREMA-D


Git LFS initialized.
Cloning into '/content/CREMA-D'...
remote: Enumerating objects: 22366, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 22366 (delta 5), reused 0 (delta 0), pack-reused 22351 (from 1)
Receiving objects: 100% (22366/22366), 13.09 MiB | 1.81 MiB/s, done.
Resolving deltas: 100% (18/18), done.
Updating files: 100% (22341/22341), done.
Filtering content: 100% (22326/22326), 3.42 GiB | 2.26 MiB/s, done.


In [ ]:

from pathlib import Path

crema_dir = Path("/content/CREMA-D")

wav_files = list(crema_dir.rglob("*.wav"))

print("WAV files found:", len(wav_files))
print("\nExample files:")
for f in wav_files[:5]:
    print(f)


WAV files found: 7442

Example files:
/content/CREMA-D/AudioWAV/1075_IEO_SAD_HI.wav
/content/CREMA-D/AudioWAV/1041_IWW_FEA_XX.wav
/content/CREMA-D/AudioWAV/1011_IOM_FEA_XX.wav
/content/CREMA-D/AudioWAV/1016_TAI_HAP_XX.wav
/content/CREMA-D/AudioWAV/1018_IWL_HAP_XX.wav


In [ ]:

from collections import Counter
from pathlib import Path

crema_dir = Path("/content/CREMA-D/AudioWAV")
wav_files = list(crema_dir.glob("*.wav"))

emotion_counts = Counter(
    f.stem.split("_")[2]
    for f in wav_files
    if len(f.stem.split("_")) >= 4
)

emotion_names = {
    "ANG": "angry",
    "DIS": "disgust",
    "FEA": "fearful",
    "HAP": "happy",
    "NEU": "neutral",
    "SAD": "sad",
}

print("Total WAV files:", len(wav_files))
print("\nEmotion distribution:")
for code, count in sorted(emotion_counts.items()):
    print(f"{emotion_names.get(code, code)} ({code}): {count}")


Total WAV files: 7442

Emotion distribution:
angry (ANG): 1271
disgust (DIS): 1271
fearful (FEA): 1271
happy (HAP): 1271
neutral (NEU): 1087
sad (SAD): 1271


### CREMA-D Dataset Verification

- **Total audio recordings:** 7,442 WAV files
- **Actors:** 91, according to the dataset README
- **Emotion classes:** Angry, disgust, fearful, happy, neutral, and sad
- **Label source:** Emotion codes encoded in the filenames
- **Storage location:** `/content/CREMA-D/AudioWAV`

The downloaded audio files were verified, and the emotion distribution was counted programmatically. The dataset contains 1,271 recordings each for angry, disgust, fearful, happy, and sad, and 1,087 recordings for neutral.

**Observation:** CREMA-D does not contain calm or surprised labels, so we will not invent those labels for its recordings.


In [ ]:

import requests

url = "https://tspace.library.utoronto.ca/handle/1807/24487"

try:
    response = requests.get(url, timeout=20)
    print("HTTP status:", response.status_code)
    print("Page accessible:", response.ok)
    print("Final URL:", response.url)
except Exception as e:
    print("Request failed:", e)


HTTP status: 200
Page accessible: True
Final URL: https://utoronto.scholaris.ca/collections/036db644-9790-4ed0-90cc-be1dfb8a4b66


In [ ]:

import requests

url = "https://utoronto.scholaris.ca/collections/036db644-9790-4ed0-90cc-be1dfb8a4b66"
response = requests.get(url, timeout=20)

print(response.text[:5000])


<!DOCTYPE html><html data-beasties-container="" lang="en"><head>
  <meta charset="UTF-8">
  <base href="/">
  <title>Toronto emotional speech set (TESS)</title>
  <meta name="viewport" content="width=device-width,minimum-scale=1">
  <meta http-equiv="cache-control" content="no-store">
<link rel="stylesheet" href="styles.cf390c6bfbe7555f.css" media="print" onload="this.media='all'"><noscript>&lt;link rel="stylesheet" href="styles.cf390c6bfbe7555f.css"&gt;</noscript><script id="hashed-file-mapping" type="application/json">{"assets/config.json":"assets/config.3062f2533676367b3ca509b37e47fb7a.json","toronto-theme.css":"toronto-theme.f0540de44211fece34088c9863e3459b.css"}</script><link rel="preload" as="fetch" href="assets/config.3062f2533676367b3ca509b37e47fb7a.json" crossorigin="anonymous" class="hfm"><link rel="prefetch" as="style" href="toronto-theme.f0540de44211fece34088c9863e3459b.css" class="hfm"><link rel="stylesheet" type="text/css" class="theme-css" href="toronto-theme.f0540de4421

In [ ]:

import requests

base = "https://utoronto.scholaris.ca"
collection_id = "036db644-9790-4ed0-90cc-be1dfb8a4b66"

response = requests.get(
    f"{base}/server/api/core/collections/{collection_id}/items",
    params={"size": 20},
    timeout=30
)

print("HTTP status:", response.status_code)

if response.ok:
    data = response.json()
    for item in data.get("_embedded", {}).get("items", []):
        print("\nTitle:", item.get("name"))
        print("UUID:", item.get("uuid"))
        print("Links:", item.get("_links", {}).keys())
else:
    print(response.text[:1000])


HTTP status: 404
{"timestamp":"2026-10-15T17:34:45.320+00:00","status":404,"error":"Not Found","message":"An exception has occurred","path":"/server/api/core/collections/036db644-9790-4ed0-90cc-be1dfb8a4b66/items"}


In [ ]:

import requests

url = "https://borealisdata.ca/api/access/dataset/:persistentId/"
params = {"persistentId": "doi:10.5683/SP2/E8H2MF"}

response = requests.get(
    url,
    params=params,
    stream=True,
    timeout=60
)

print("HTTP status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Content length:", response.headers.get("Content-Length"))

if response.ok and "zip" in response.headers.get("Content-Type", "").lower():
    output_path = "/content/TESS.zip"

    with open(output_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    print("Saved dataset to:", output_path)
else:
    print("Not saved. Response preview:")
    response.close()
    check = requests.get(url, params=params, timeout=30)
    print(check.text[:1000])


HTTP status: 200
Content type: application/zip;charset=UTF-8
Content length: None
Saved dataset to: /content/TESS.zip


In [ ]:


import zipfile
from pathlib import Path

zip_path = Path("/content/TESS.zip")

print("ZIP size (MB):", round(zip_path.stat().st_size / (1024 * 1024), 2))

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()
    audio_files = [
        name for name in files
        if name.lower().endswith((".wav", ".mp3", ".flac"))
    ]

    print("Total entries:", len(files))
    print("Audio files:", len(audio_files))
    print("\nExample audio paths:")
    for name in audio_files[:10]:
        print(name)


ZIP size (MB): 213.46
Total entries: 2801
Audio files: 2800

Example audio paths:
OAF_back_angry.wav
OAF_back_disgust.wav
OAF_back_fear.wav
OAF_back_happy.wav
OAF_back_neutral.wav
OAF_back_ps.wav
OAF_back_sad.wav
OAF_bar_angry.wav
OAF_bar_disgust.wav
OAF_bar_fear.wav


In [ ]:

import zipfile
from collections import Counter
from pathlib import PurePosixPath

zip_path = "/content/TESS.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    audio_files = [
        name for name in z.namelist()
        if name.lower().endswith(".wav")
    ]

emotion_counts = Counter()

for name in audio_files:
    filename = PurePosixPath(name).stem.lower()
    emotion = filename.split("_")[-1]

    if emotion in {"angry", "disgust", "fear", "happy",
                   "neutral", "ps", "sad"}:
        emotion_counts[emotion] += 1
    else:
        print("Unrecognized label:", name)

label_names = {
    "angry": "angry",
    "disgust": "disgust",
    "fear": "fearful",
    "happy": "happy",
    "neutral": "neutral",
    "ps": "surprised (pleasant surprise)",
    "sad": "sad",
}

print("Total audio files:", len(audio_files))
print("\nEmotion distribution:")

for label, count in sorted(emotion_counts.items()):
    print(f"{label_names[label]}: {count}")

print("\nTotal counted:", sum(emotion_counts.values()))


Total audio files: 2800

Emotion distribution:
angry: 400
disgust: 400
fearful: 400
happy: 400
neutral: 400
surprised (pleasant surprise): 400
sad: 400

Total counted: 2800


### TESS Dataset Verification

- **Total audio recordings:** 2,800 WAV files
- **Speakers:** 2 actresses
- **Emotion classes:** Angry, disgust, fearful, happy, neutral, surprised (pleasant surprise), and sad
- **Recordings per emotion:** 400
- **Label source:** Emotion codes in the audio filenames
- **Storage location:** `/content/TESS.zip`

**Observation:** TESS has a balanced distribution, with an equal number of recordings for each of its seven emotion classes. The pleasant-surprise label is mapped to `surprised` in our common label set.

In [ ]:
import os

for path in ["/content/SAVEE", "/content/SAVEE.zip"]:
    print(path, "exists:", os.path.exists(path))


/content/SAVEE exists: False
/content/SAVEE.zip exists: False


### Dataset Selection and Availability

Three datasets are currently included in the investigation: CREMA-D, RAVDESS, and TESS. SAVEE was considered but excluded because its download page was unavailable during this stage of the project.

The selected datasets contain different emotion categories and have different speaker distributions. Before combining them, emotion labels will be mapped to a common label set, and the data will be split by speaker where possible to reduce speaker leakage between training and evaluation sets.

**Current common emotion labels:** angry, calm, disgust, fearful, happy, neutral, sad, surprised.

In [ ]:
import os

print("CREMA-D:", os.path.exists("/content/CREMA-D/AudioWAV"))
print("TESS:", os.path.exists("/content/TESS.zip"))
print("RAVDESS archive:", os.path.exists("/content/drive/MyDrive/SpeechEmotionProject/ravdess.zip"))

CREMA-D: True
TESS: True
RAVDESS archive: True


In [ ]:
import zipfile
from collections import Counter
from pathlib import PurePosixPath

ravdess_path = "/content/drive/MyDrive/SpeechEmotionProject/ravdess.zip"

with zipfile.ZipFile(ravdess_path, "r") as z:
    audio_files = [
        name for name in z.namelist()
        if name.lower().endswith(".wav")
        and not PurePosixPath(name).name.startswith("._")
    ]

print("Total WAV entries:", len(audio_files))
print("Example filenames:")
for name in audio_files[:10]:
    print(PurePosixPath(name).name)

Total WAV entries: 2880
Example filenames:
03-01-01-01-01-01-01.wav
03-01-01-01-01-02-01.wav
03-01-01-01-02-01-01.wav
03-01-01-01-02-02-01.wav
03-01-02-01-01-01-01.wav
03-01-02-01-01-02-01.wav
03-01-02-01-02-01-01.wav
03-01-02-01-02-02-01.wav
03-01-02-02-01-01-01.wav
03-01-02-02-01-02-01.wav


In [ ]:
import zipfile
from collections import Counter
from pathlib import PurePosixPath

ravdess_path = "/content/drive/MyDrive/SpeechEmotionProject/ravdess.zip"

with zipfile.ZipFile(ravdess_path, "r") as z:
    files = [
        name for name in z.namelist()
        if name.lower().endswith(".wav")
        and not PurePosixPath(name).name.startswith("._")
    ]

# RAVDESS filenames encode emotion in the third field.
# Example: 03-01-05-01-01-01-01.wav
emotion_codes = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised",
}

counts = Counter()
unique_names = set()

for path in files:
    filename = PurePosixPath(path).name
    parts = filename.removesuffix(".wav").split("-")

    if len(parts) == 7 and parts[2] in emotion_codes:
        counts[emotion_codes[parts[2]]] += 1
        unique_names.add(filename)

print("Total WAV entries:", len(files))
print("Unique WAV filenames:", len(unique_names))
print("\nEmotion counts (including duplicate entries):")

for emotion, count in sorted(counts.items()):
    print(f"{emotion}: {count}")

print("\nTotal classified:", sum(counts.values()))

Total WAV entries: 2880
Unique WAV filenames: 1440

Emotion counts (including duplicate entries):
angry: 384
calm: 384
disgust: 384
fearful: 384
happy: 384
neutral: 192
sad: 384
surprised: 384

Total classified: 2880


### RAVDESS Dataset Verification

- **Total WAV entries in archive:** 2,880
- **Unique WAV filenames:** 1,440
- **Speakers:** 24 actors
- **Emotion classes:** Angry, calm, disgust, fearful, happy, neutral, sad, surprised
- **Neutral recordings:** 96
- **Recordings for each other emotion:** 192
- **Label source:** Third field of the RAVDESS filename
- **Storage location:** Google Drive, `SpeechEmotionProject/ravdess.zip`

**Observation:** The archive contains duplicate filename entries. The dataset inventory will use unique recordings to avoid counting the same filename more than once.

## Emotion Label Standardization

The CREMA-D, RAVDESS, and TESS datasets use different naming conventions for some emotions. To make them compatible, their labels will be mapped to a shared set of eight target classes.

| Dataset label | Standard label |
|---|---|
| anger / angry | angry |
| disgust | disgust |
| fear / fearful | fearful |
| happy / happiness | happy |
| neutral | neutral |
| calm | calm |
| sad / sadness | sad |
| surprise / pleasant surprise | surprised |

**Important:** A dataset does not need to contain all eight classes. We will only map emotions that are actually present and will not invent missing labels.

In [ ]:
# Standard emotion labels used by our project

emotion_mapping = {
    # CREMA-D
    "ANG": "angry",
    "DIS": "disgust",
    "FEA": "fearful",
    "HAP": "happy",
    "NEU": "neutral",
    "SAD": "sad",

    # TESS
    "angry": "angry",
    "disgust": "disgust",
    "fear": "fearful",
    "happy": "happy",
    "neutral": "neutral",
    "ps": "surprised",
    "sad": "sad",

    # RAVDESS emotion codes
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised",
}

print("Standard emotion classes:")
print(sorted(set(emotion_mapping.values())))

Standard emotion classes:
['angry', 'calm', 'disgust', 'fearful', 'happy', 'neutral', 'sad', 'surprised']


In [ ]:
import zipfile
from pathlib import PurePosixPath

ravdess_path = "/content/drive/MyDrive/SpeechEmotionProject/ravdess.zip"

with zipfile.ZipFile(ravdess_path, "r") as z:
    files = sorted({
        PurePosixPath(name).name
        for name in z.namelist()
        if name.lower().endswith(".wav")
        and not PurePosixPath(name).name.startswith("._")
    })

actors = set()

for filename in files:
    parts = filename.removesuffix(".wav").split("-")
    if len(parts) == 7:
        actors.add(parts[6])

print("Unique recordings:", len(files))
print("Unique actor IDs:", len(actors))
print("Actor IDs:", sorted(actors))

Unique recordings: 1440
Unique actor IDs: 24
Actor IDs: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24']


In [ ]:
import zipfile
from collections import Counter
from pathlib import PurePosixPath

ravdess_path = "/content/drive/MyDrive/SpeechEmotionProject/ravdess.zip"

with zipfile.ZipFile(ravdess_path, "r") as z:
    wav_paths = [
        name for name in z.namelist()
        if name.lower().endswith(".wav")
        and not PurePosixPath(name).name.startswith("._")
    ]

counts = Counter(PurePosixPath(name).name for name in wav_paths)

print("Archive WAV entries:", len(wav_paths))
print("Unique filenames:", len(counts))
print("Filenames occurring more than once:", sum(n > 1 for n in counts.values()))
print("Extra repeated entries:", sum(n - 1 for n in counts.values()))

Archive WAV entries: 2880
Unique filenames: 1440
Filenames occurring more than once: 1440
Extra repeated entries: 1440


## Speaker-Independent Data Splitting Strategy

To evaluate generalization to unseen speakers, recordings will be divided by speaker rather than randomly by individual audio file.

- **Training set:** used to learn model parameters.
- **Validation set:** used to monitor training and select the model.
- **Test set:** reserved for final evaluation.

All recordings belonging to a particular speaker must remain in the same split. This prevents recordings from the same speaker from appearing in both training and evaluation sets.

RAVDESS contains 24 actors, CREMA-D contains 91 actors, and TESS contains 2 speakers. Because TESS has only two speakers, its speaker-independent evaluation will require special consideration.

Duplicate filename entries in the RAVDESS archive will be deduplicated before preparing the dataset.

In [ ]:
import os

base = "/content/drive/MyDrive/SpeechEmotionProject"

for folder in ["final_model", "final_model_v2"]:
    path = os.path.join(base, folder)
    print(f"\n{folder}:")
    if os.path.isdir(path):
        for name in os.listdir(path):
            print(" -", name)
    else:
        print(" Folder not found")


final_model:
 - config.json
 - model.safetensors
 - training_args.bin
 - preprocessor_config.json

final_model_v2:
 - config.json
 - model.safetensors
 - training_args.bin
 - preprocessor_config.json


In [ ]:
import json
import os

base = "/content/drive/MyDrive/SpeechEmotionProject"

for folder in ["final_model", "final_model_v2"]:
    config_path = os.path.join(base, folder, "config.json")

    with open(config_path, "r") as f:
        config = json.load(f)

    print(f"\n--- {folder} ---")
    print("Model architecture:", config.get("model_type"))
    print("Base model:", config.get("_name_or_path"))
    print("Number of labels:", config.get("num_labels"))
    print("Label mapping:", config.get("id2label"))


--- final_model ---
Model architecture: wav2vec2
Base model: None
Number of labels: None
Label mapping: {'0': 'angry', '1': 'calm', '2': 'disgust', '3': 'fearful', '4': 'happy', '5': 'neutral', '6': 'sad', '7': 'surprised'}

--- final_model_v2 ---
Model architecture: wav2vec2
Base model: None
Number of labels: None
Label mapping: {'0': 'angry', '1': 'calm', '2': 'disgust', '3': 'fearful', '4': 'happy', '5': 'neutral', '6': 'sad', '7': 'surprised'}


In [ ]:
import os
import torch

base = "/content/drive/MyDrive/SpeechEmotionProject"

for folder in ["final_model", "final_model_v2"]:
    path = os.path.join(base, folder, "training_args.bin")
    print(f"\n--- {folder} ---")

    try:
        args = torch.load(path, map_location="cpu", weights_only=False)
        for key in [
            "num_train_epochs",
            "learning_rate",
            "per_device_train_batch_size",
            "gradient_accumulation_steps",
            "fp16",
            "output_dir",
        ]:
            print(f"{key}: {getattr(args, key, 'not recorded')}")
    except Exception as e:
        print("Could not read training arguments:", repr(e))


--- final_model ---
num_train_epochs: 5
learning_rate: 1e-05
per_device_train_batch_size: 4
gradient_accumulation_steps: 2
fp16: True
output_dir: /content/drive/MyDrive/SpeechEmotionProject/checkpoints

--- final_model_v2 ---
num_train_epochs: 3
learning_rate: 2e-06
per_device_train_batch_size: 4
gradient_accumulation_steps: 2
fp16: True
output_dir: /content/drive/MyDrive/SpeechEmotionProject/experiment_2


In [ ]:
import os
import json

base = "/content/drive/MyDrive/SpeechEmotionProject"

for folder in ["final_model", "final_model_v2"]:
    config_path = os.path.join(base, folder, "config.json")

    with open(config_path, "r") as f:
        config = json.load(f)

    print(f"\n--- {folder} ---")
    print("Architecture:", config.get("architectures"))
    print("Model type:", config.get("model_type"))
    print("Hidden size:", config.get("hidden_size"))
    print("Number of labels:", len(config.get("id2label", {})))
    print("Labels:", config.get("id2label"))


--- final_model ---
Architecture: ['Wav2Vec2ForSequenceClassification']
Model type: wav2vec2
Hidden size: 768
Number of labels: 8
Labels: {'0': 'angry', '1': 'calm', '2': 'disgust', '3': 'fearful', '4': 'happy', '5': 'neutral', '6': 'sad', '7': 'surprised'}

--- final_model_v2 ---
Architecture: ['Wav2Vec2ForSequenceClassification']
Model type: wav2vec2
Hidden size: 768
Number of labels: 8
Labels: {'0': 'angry', '1': 'calm', '2': 'disgust', '3': 'fearful', '4': 'happy', '5': 'neutral', '6': 'sad', '7': 'surprised'}


In [ ]:
import os
import zipfile
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/SpeechEmotionProject")
RAVDESS_ZIP = BASE / "ravdess.zip"
CREMA_DIR = Path("/content/CREMA-D/AudioWAV")
TESS_ZIP = Path("/content/TESS.zip")

RAVDESS_DIR = Path("/content/ravdess_extracted")
TESS_DIR = Path("/content/tess_extracted")

# Extract only if the folders are not already populated.
def extract_if_needed(zip_path, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    if not any(output_dir.rglob("*.wav")):
        print(f"Extracting {zip_path.name}...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(output_dir)
    else:
        print(f"Already extracted: {output_dir}")

extract_if_needed(RAVDESS_ZIP, RAVDESS_DIR)
extract_if_needed(TESS_ZIP, TESS_DIR)

# Standard labels shared by our project
emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised",
}

records = []
seen_ravdess = set()

# RAVDESS: filename format includes emotion and actor IDs.
for p in RAVDESS_DIR.rglob("*.wav"):
    parts = p.stem.split("-")
    if len(parts) >= 7 and parts[2] in emotion_map:
        # Duplicate archive entries should not count twice.
        key = p.name
        if key in seen_ravdess:
            continue
        seen_ravdess.add(key)
        records.append({
            "path": str(p),
            "dataset": "ravdess",
            "emotion": emotion_map[parts[2]],
            "speaker": f"ravdess_{parts[6]}",
        })

# CREMA-D: e.g. 1001_DFA_ANG_XX.wav
crema_map = {
    "ANG": "angry", "DIS": "disgust", "FEA": "fearful",
    "HAP": "happy", "NEU": "neutral", "SAD": "sad",
}

for p in CREMA_DIR.rglob("*.wav"):
    parts = p.stem.split("_")
    if len(parts) >= 3 and parts[2] in crema_map:
        records.append({
            "path": str(p),
            "dataset": "crema",
            "emotion": crema_map[parts[2]],
            "speaker": f"crema_{parts[0]}",
        })

# TESS: infer emotion from the containing folder or filename.
tess_map = {
    "angry": "angry",
    "disgust": "disgust",
    "fear": "fearful",
    "fearful": "fearful",
    "happy": "happy",
    "neutral": "neutral",
    "ps": "surprised",
    "sad": "sad",
}

for p in TESS_DIR.rglob("*.wav"):
    parts = [part.lower() for part in p.parts]
    emotion = None

    for part in reversed(parts):
        for code, label in tess_map.items():
            if part == code or part.endswith("_" + code):
                emotion = label
                break
        if emotion:
            break

    if emotion:
        # TESS recordings identify the speaker as OAF or YAF.
        speaker_token = p.name[:3].upper()
        if speaker_token not in {"OAF", "YAF"}:
            speaker_token = p.parent.name.split("_")[0].upper()

        records.append({
            "path": str(p),
            "dataset": "tess",
            "emotion": emotion,
            "speaker": f"tess_{speaker_token}",
        })

df = pd.DataFrame(records).drop_duplicates(subset=["path"]).reset_index(drop=True)

print("Total indexed recordings:", len(df))
print("\nRecordings by dataset:")
print(df["dataset"].value_counts())
print("\nRecordings by emotion:")
print(df["emotion"].value_counts().sort_index())
print("\nSpeakers by dataset:")
print(df.groupby("dataset")["speaker"].nunique())
print("\nExample records:")
display(df.head())

Extracting ravdess.zip...
Extracting TESS.zip...
Total indexed recordings: 8882

Recordings by dataset:
dataset
crema      7442
ravdess    1440
Name: count, dtype: int64

Recordings by emotion:
emotion
angry        1463
calm          192
disgust      1463
fearful      1463
happy        1463
neutral      1183
sad          1463
surprised     192
Name: count, dtype: int64

Speakers by dataset:
dataset
crema      91
ravdess    24
Name: speaker, dtype: int64

Example records:


,path,dataset,emotion,speaker
0,/content/ravdess_extracted/Actor_02/03-01-07-0...,ravdess,disgust,ravdess_02
1,/content/ravdess_extracted/Actor_02/03-01-06-0...,ravdess,fearful,ravdess_02
2,/content/ravdess_extracted/Actor_02/03-01-05-0...,ravdess,angry,ravdess_02
3,/content/ravdess_extracted/Actor_02/03-01-03-0...,ravdess,happy,ravdess_02
4,/content/ravdess_extracted/Actor_02/03-01-01-0...,ravdess,neutral,ravdess_02


In [ ]:
from pathlib import Path

TESS_DIR = Path("/content/tess_extracted")

wav_files = list(TESS_DIR.rglob("*.wav"))

print("TESS WAV files found:", len(wav_files))
print("\nExample paths:")

for p in wav_files[:12]:
    print(p)

TESS WAV files found: 2800

Example paths:
/content/tess_extracted/OAF_shack_ps.wav
/content/tess_extracted/YAF_juice_fear.wav
/content/tess_extracted/YAF_tip_happy.wav
/content/tess_extracted/OAF_jug_angry.wav
/content/tess_extracted/YAF_pole_happy.wav
/content/tess_extracted/OAF_laud_happy.wav
/content/tess_extracted/OAF_get_angry.wav
/content/tess_extracted/OAF_red_fear.wav
/content/tess_extracted/OAF_far_neutral.wav
/content/tess_extracted/OAF_walk_ps.wav
/content/tess_extracted/OAF_such_fear.wav
/content/tess_extracted/OAF_check_sad.wav


In [ ]:
from pathlib import Path
import pandas as pd

TESS_DIR = Path("/content/tess_extracted")

tess_emotion_map = {
    "angry": "angry",
    "disgust": "disgust",
    "fear": "fearful",
    "happy": "happy",
    "neutral": "neutral",
    "ps": "surprised",
    "sad": "sad",
}

tess_records = []

for p in TESS_DIR.rglob("*.wav"):
    parts = p.stem.split("_")

    if len(parts) < 3:
        continue

    speaker_id = parts[0].upper()
    emotion_code = parts[-1].lower()

    if emotion_code not in tess_emotion_map:
        continue

    tess_records.append({
        "path": str(p),
        "dataset": "tess",
        "emotion": tess_emotion_map[emotion_code],
        "speaker": f"tess_{speaker_id}",
    })

tess_df = pd.DataFrame(tess_records)

# Remove any previously indexed TESS rows before adding them.
df = df[df["dataset"] != "tess"].copy()
df = pd.concat([df, tess_df], ignore_index=True)
df = df.drop_duplicates(subset=["path"]).reset_index(drop=True)

print("Total indexed recordings:", len(df))
print("\nRecordings by dataset:")
print(df["dataset"].value_counts())

print("\nRecordings by emotion:")
print(df["emotion"].value_counts().sort_index())

print("\nSpeakers by dataset:")
print(df.groupby("dataset")["speaker"].nunique())

Total indexed recordings: 11682

Recordings by dataset:
dataset
crema      7442
tess       2800
ravdess    1440
Name: count, dtype: int64

Recordings by emotion:
emotion
angry        1863
calm          192
disgust      1863
fearful      1863
happy        1863
neutral      1583
sad          1863
surprised     592
Name: count, dtype: int64

Speakers by dataset:
dataset
crema      91
ravdess    24
tess        2
Name: speaker, dtype: int64


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Separate TESS for external evaluation.
trainval_df = df[df["dataset"].isin(["crema", "ravdess"])].copy()
tess_test_df = df[df["dataset"] == "tess"].copy()

# Split speakers, not individual recordings.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        trainval_df,
        groups=trainval_df["speaker"]
    )
)

train_df = trainval_df.iloc[train_idx].reset_index(drop=True)
val_df = trainval_df.iloc[val_idx].reset_index(drop=True)
tess_test_df = tess_test_df.reset_index(drop=True)

# Verify there is no speaker overlap.
train_speakers = set(train_df["speaker"])
val_speakers = set(val_df["speaker"])

print("Train recordings:", len(train_df))
print("Validation recordings:", len(val_df))
print("External TESS recordings:", len(tess_test_df))

print("\nTrain/validation speaker overlap:",
      len(train_speakers & val_speakers))

print("\nTraining emotion counts:")
print(train_df["emotion"].value_counts().sort_index())

print("\nValidation emotion counts:")
print(val_df["emotion"].value_counts().sort_index())

print("\nTESS speaker counts:")
print(tess_test_df["speaker"].value_counts())

Train recordings: 7112
Validation recordings: 1770
External TESS recordings: 2800

Train/validation speaker overlap: 0

Training emotion counts:
emotion
angry        1172
calm          152
disgust      1172
fearful      1172
happy        1172
neutral       948
sad          1172
surprised     152
Name: count, dtype: int64

Validation emotion counts:
emotion
angry        291
calm          40
disgust      291
fearful      291
happy        291
neutral      235
sad          291
surprised     40
Name: count, dtype: int64

TESS speaker counts:
speaker
tess_OAF    1400
tess_YAF    1400
Name: count, dtype: int64


In [1]:
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("External test:", len(tess_test_df))

NameError: name 'train_df' is not defined

In [2]:
import os

paths = {
    "Google Drive": "/content/drive/MyDrive/SpeechEmotionProject",
    "CREMA-D": "/content/CREMA-D/AudioWAV",
    "TESS archive": "/content/TESS.zip",
    "TESS extracted": "/content/tess_extracted",
    "RAVDESS extracted": "/content/ravdess_extracted",
}

for name, path in paths.items():
    print(f"{name}: {'FOUND' if os.path.exists(path) else 'NOT FOUND'}")

Google Drive: NOT FOUND
CREMA-D: NOT FOUND
TESS archive: NOT FOUND
TESS extracted: NOT FOUND
RAVDESS extracted: NOT FOUND


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

base = "/content/drive/MyDrive/SpeechEmotionProject"

print("Project folder exists:", os.path.exists(base))

if os.path.exists(base):
    print("Files and folders:")
    print(os.listdir(base))

Project folder exists: True
Files and folders:
['ravdess.zip', 'checkpoints', 'final_model', 'experiment_2', 'final_model_v2']


In [5]:
import os
import zipfile
import subprocess
import urllib.request

# 1. Download CREMA-D
if not os.path.exists("/content/CREMA-D/AudioWAV"):
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/CheyneyComputerScience/CREMA-D.git",
        "/content/CREMA-D"
    ], check=True)

print("CREMA-D recordings:",
      len([f for f in os.listdir("/content/CREMA-D/AudioWAV")
           if f.lower().endswith(".wav")]))

# 2. Download and extract TESS
tess_zip = "/content/TESS.zip"
tess_dir = "/content/tess_extracted"

if not os.path.exists(tess_zip):
    urllib.request.urlretrieve(
        "https://tessdata.s3.amazonaws.com/TESS Toronto emotional speech set data.zip",
        tess_zip
    )

if not os.path.exists(tess_dir):
    os.makedirs(tess_dir, exist_ok=True)
    with zipfile.ZipFile(tess_zip, "r") as z:
        z.extractall(tess_dir)

print("TESS downloaded and extracted.")

CREMA-D recordings: 7442


InvalidURL: URL can't contain control characters. '/TESS Toronto emotional speech set data.zip' (found at least ' ')

In [6]:
import os
import zipfile
import urllib.request

tess_zip = "/content/TESS.zip"
tess_dir = "/content/tess_extracted"

url = "https://tessdata.s3.amazonaws.com/TESS%20Toronto%20emotional%20speech%20set%20data.zip"

if not os.path.exists(tess_zip):
    urllib.request.urlretrieve(url, tess_zip)

os.makedirs(tess_dir, exist_ok=True)

with zipfile.ZipFile(tess_zip, "r") as z:
    z.extractall(tess_dir)

print("TESS extracted successfully.")
print("WAV recordings:", sum(
    1 for root, _, files in os.walk(tess_dir)
    for f in files if f.lower().endswith(".wav")
))

HTTPError: HTTP Error 403: Forbidden

In [7]:
import os
import requests
import zipfile

tess_zip = "/content/TESS.zip"
tess_dir = "/content/tess_extracted"

url = (
    "https://borealisdata.ca/api/access/dataset/:persistentId"
    "?persistentId=doi:10.5683/SP2/E8H2MF"
)

response = requests.get(url, timeout=120)
print("HTTP status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Downloaded bytes:", len(response.content))

if response.ok and response.content[:2] == b"PK":
    with open(tess_zip, "wb") as f:
        f.write(response.content)

    os.makedirs(tess_dir, exist_ok=True)
    with zipfile.ZipFile(tess_zip, "r") as z:
        z.extractall(tess_dir)

    count = sum(
        1 for root, _, files in os.walk(tess_dir)
        for name in files if name.lower().endswith(".wav")
    )
    print("TESS WAV recordings:", count)
else:
    print("The response wasn't a ZIP archive. No file was extracted.")

HTTP status: 200
Content type: application/zip;charset=UTF-8
Downloaded bytes: 223829029
TESS WAV recordings: 2800


In [8]:
import os
import zipfile
import pandas as pd

# Extract RAVDESS from Google Drive
ravdess_zip = "/content/drive/MyDrive/SpeechEmotionProject/ravdess.zip"
ravdess_dir = "/content/ravdess_extracted"

if not os.path.exists(ravdess_dir):
    os.makedirs(ravdess_dir, exist_ok=True)
    with zipfile.ZipFile(ravdess_zip, "r") as z:
        z.extractall(ravdess_dir)

# Find audio files
crema_dir = "/content/CREMA-D/AudioWAV"
tess_dir = "/content/tess_extracted"

records = []

# CREMA-D labels
crema_labels = {
    "ANG": "angry", "DIS": "disgust", "FEA": "fearful",
    "HAP": "happy", "NEU": "neutral", "SAD": "sad"
}

for f in os.listdir(crema_dir):
    if f.lower().endswith(".wav"):
        parts = f.split("_")
        if len(parts) >= 3 and parts[2] in crema_labels:
            records.append({
                "path": os.path.join(crema_dir, f),
                "dataset": "crema",
                "emotion": crema_labels[parts[2]],
                "speaker": "crema_" + parts[0]
            })

# TESS labels
tess_labels = {
    "angry": "angry", "disgust": "disgust",
    "fear": "fearful", "happy": "happy",
    "neutral": "neutral", "ps": "surprised",
    "sad": "sad"
}

for root, _, files in os.walk(tess_dir):
    for f in files:
        if f.lower().endswith(".wav"):
            parts = f.replace(".wav", "").split("_")
            code = parts[-1].lower()
            if code in tess_labels:
                records.append({
                    "path": os.path.join(root, f),
                    "dataset": "tess",
                    "emotion": tess_labels[code],
                    "speaker": "tess_" + parts[0]
                })

# RAVDESS labels
ravdess_labels = {
    "01": "neutral", "02": "calm", "03": "happy",
    "04": "sad", "05": "angry", "06": "fearful",
    "07": "disgust", "08": "surprised"
}

for root, _, files in os.walk(ravdess_dir):
    for f in files:
        if f.lower().endswith(".wav"):
            parts = f.split("-")
            if len(parts) == 7 and parts[2] in ravdess_labels:
                records.append({
                    "path": os.path.join(root, f),
                    "dataset": "ravdess",
                    "emotion": ravdess_labels[parts[2]],
                    "speaker": "ravdess_" + parts[6].replace(".wav", "")
                })

df = pd.DataFrame(records).drop_duplicates(subset="path").reset_index(drop=True)

print("Total recordings:", len(df))
print("\nBy dataset:")
print(df["dataset"].value_counts())
print("\nBy emotion:")
print(df["emotion"].value_counts())

Total recordings: 13122

By dataset:
dataset
crema      7442
ravdess    2880
tess       2800
Name: count, dtype: int64

By emotion:
emotion
sad          2055
fearful      2055
happy        2055
disgust      2055
angry        2055
neutral      1679
surprised     784
calm          384
Name: count, dtype: int64


In [9]:
from collections import Counter
import os

ravdess_dir = "/content/ravdess_extracted"
categories = Counter()
wav_count = 0

for root, _, files in os.walk(ravdess_dir):
    for f in files:
        if f.lower().endswith(".wav"):
            parts = f.split("-")
            if len(parts) == 7:
                categories[(parts[1], parts[4])] += 1
                wav_count += 1

print("Total RAVDESS WAV files:", wav_count)
print("Modality and vocal-channel counts:")
for category, count in sorted(categories.items()):
    print(category, count)

Total RAVDESS WAV files: 2880
Modality and vocal-channel counts:
('01', '01') 1440
('01', '02') 1440


In [10]:
from collections import Counter
import os

ravdess_dir = "/content/ravdess_extracted"
counts = Counter()

for root, _, files in os.walk(ravdess_dir):
    for f in files:
        if f.lower().endswith(".wav"):
            parts = f.replace(".wav", "").split("-")
            if len(parts) == 7:
                modality = parts[1]
                vocal_channel = parts[2]
                emotion = parts[3]
                counts[(modality, vocal_channel, emotion)] += 1

print("Modality | Vocal channel | Emotion code | Count")
for key, count in sorted(counts.items()):
    print(*key, count, sep=" | ")

Modality | Vocal channel | Emotion code | Count
01 | 01 | 01 | 192
01 | 02 | 01 | 192
01 | 02 | 02 | 192
01 | 03 | 01 | 192
01 | 03 | 02 | 192
01 | 04 | 01 | 192
01 | 04 | 02 | 192
01 | 05 | 01 | 192
01 | 05 | 02 | 192
01 | 06 | 01 | 192
01 | 06 | 02 | 192
01 | 07 | 01 | 192
01 | 07 | 02 | 192
01 | 08 | 01 | 192
01 | 08 | 02 | 192


In [11]:
import os
from collections import Counter

ravdess_dir = "/content/ravdess_extracted"
counts = Counter()

for root, _, files in os.walk(ravdess_dir):
    for f in files:
        if f.lower().endswith(".wav"):
            parts = f.replace(".wav", "").split("-")
            if len(parts) == 7:
                counts[parts[0]] += 1

print("Filename prefix counts:", counts)
print("Example filenames:")
shown = 0
for root, _, files in os.walk(ravdess_dir):
    for f in files:
        if f.lower().endswith(".wav"):
            print(f)
            shown += 1
            if shown == 5:
                break
    if shown == 5:
        break

Filename prefix counts: Counter({'03': 2880})
Example filenames:
03-01-07-01-02-02-02.wav
03-01-06-02-02-02-02.wav
03-01-05-01-02-02-02.wav
03-01-03-01-01-01-02.wav
03-01-01-01-01-01-02.wav


In [12]:
import os
import pandas as pd

records = []

# CREMA-D
crema_dir = "/content/CREMA-D/AudioWAV"
crema_labels = {
    "ANG": "angry", "DIS": "disgust", "FEA": "fearful",
    "HAP": "happy", "NEU": "neutral", "SAD": "sad"
}

for f in os.listdir(crema_dir):
    if f.lower().endswith(".wav"):
        parts = f.split("_")
        if len(parts) >= 3 and parts[2] in crema_labels:
            records.append({
                "path": os.path.join(crema_dir, f),
                "dataset": "crema",
                "emotion": crema_labels[parts[2]],
                "speaker": "crema_" + parts[0]
            })

# TESS
tess_dir = "/content/tess_extracted"
tess_labels = {
    "angry": "angry", "disgust": "disgust", "fear": "fearful",
    "happy": "happy", "neutral": "neutral",
    "ps": "surprised", "sad": "sad"
}

for root, _, files in os.walk(tess_dir):
    for f in files:
        if f.lower().endswith(".wav"):
            parts = f[:-4].split("_")
            code = parts[-1].lower()
            if code in tess_labels:
                records.append({
                    "path": os.path.join(root, f),
                    "dataset": "tess",
                    "emotion": tess_labels[code],
                    "speaker": "tess_" + parts[0]
                })

# RAVDESS: audio-only, speech, all 8 emotion codes
ravdess_dir = "/content/ravdess_extracted"
ravdess_labels = {
    "01": "neutral", "02": "calm", "03": "happy",
    "04": "sad", "05": "angry", "06": "fearful",
    "07": "disgust", "08": "surprised"
}

for root, _, files in os.walk(ravdess_dir):
    for f in files:
        if f.lower().endswith(".wav"):
            parts = f[:-4].split("-")
            if len(parts) == 7 and parts[0] == "03":
                emotion_code = parts[2]
                statement_code = parts[4]
                if emotion_code in ravdess_labels and statement_code in {"01", "02"}:
                    records.append({
                        "path": os.path.join(root, f),
                        "dataset": "ravdess",
                        "emotion": ravdess_labels[emotion_code],
                        "speaker": "ravdess_" + parts[6]
                    })

df = pd.DataFrame(records).drop_duplicates(subset="path").reset_index(drop=True)

print("Total recordings:", len(df))
print("\nBy dataset:")
print(df["dataset"].value_counts())
print("\nBy emotion:")
print(df["emotion"].value_counts())

Total recordings: 13122

By dataset:
dataset
crema      7442
ravdess    2880
tess       2800
Name: count, dtype: int64

By emotion:
emotion
sad          2055
fearful      2055
happy        2055
disgust      2055
angry        2055
neutral      1679
surprised     784
calm          384
Name: count, dtype: int64


In [13]:
from collections import Counter
import os

ravdess_dir = "/content/ravdess_extracted"
counts = Counter()

for root, _, files in os.walk(ravdess_dir):
    for f in files:
        if f.lower().endswith(".wav"):
            parts = f[:-4].split("-")
            if len(parts) == 7 and parts[0] == "03":
                counts[parts[1]] += 1

print(counts)

Counter({'01': 2880})


In [14]:
import os
import hashlib
from collections import defaultdict

ravdess_dir = "/content/ravdess_extracted"
hash_to_files = defaultdict(list)

for root, _, files in os.walk(ravdess_dir):
    for f in files:
        if f.lower().endswith(".wav"):
            path = os.path.join(root, f)

            h = hashlib.sha256()
            with open(path, "rb") as audio_file:
                for chunk in iter(lambda: audio_file.read(1024 * 1024), b""):
                    h.update(chunk)

            hash_to_files[h.hexdigest()].append(path)

unique_hashes = len(hash_to_files)
duplicate_groups = [
    paths for paths in hash_to_files.values() if len(paths) > 1
]

print("Total WAV files:", sum(len(v) for v in hash_to_files.values()))
print("Unique audio files:", unique_hashes)
print("Duplicate groups:", len(duplicate_groups))
print("Extra duplicate files:",
      sum(len(v) - 1 for v in hash_to_files.values() if len(v) > 1))

Total WAV files: 2880
Unique audio files: 1439
Duplicate groups: 1439
Extra duplicate files: 1441


In [15]:
for paths in hash_to_files.values():
    if len(paths) != 2:
        print(f"Group contains {len(paths)} identical files:")
        for p in paths:
            print(" ", os.path.basename(p))

Group contains 4 identical files:
  03-01-03-01-02-02-07.wav
  03-01-03-01-02-01-07.wav
  03-01-03-01-02-02-07.wav
  03-01-03-01-02-01-07.wav


In [16]:
import os
import hashlib

ravdess_dir = "/content/ravdess_extracted"
unique_ravdess = []
seen_hashes = set()

for root, _, files in os.walk(ravdess_dir):
    for f in sorted(files):
        if not f.lower().endswith(".wav"):
            continue

        path = os.path.join(root, f)

        h = hashlib.sha256()
        with open(path, "rb") as audio_file:
            for chunk in iter(lambda: audio_file.read(1024 * 1024), b""):
                h.update(chunk)

        digest = h.hexdigest()

        if digest not in seen_hashes:
            seen_hashes.add(digest)
            unique_ravdess.append(path)

print("Unique RAVDESS recordings:", len(unique_ravdess))

Unique RAVDESS recordings: 1439


In [17]:
import os
import pandas as pd

records = []

# CREMA-D
crema_dir = "/content/CREMA-D/AudioWAV"
crema_labels = {
    "ANG": "angry", "DIS": "disgust", "FEA": "fearful",
    "HAP": "happy", "NEU": "neutral", "SAD": "sad"
}

for f in os.listdir(crema_dir):
    if f.lower().endswith(".wav"):
        parts = f.split("_")
        if len(parts) >= 3 and parts[2] in crema_labels:
            records.append({
                "path": os.path.join(crema_dir, f),
                "dataset": "crema",
                "emotion": crema_labels[parts[2]],
                "speaker": "crema_" + parts[0]
            })

# TESS
tess_dir = "/content/tess_extracted"
tess_labels = {
    "angry": "angry", "disgust": "disgust", "fear": "fearful",
    "happy": "happy", "neutral": "neutral",
    "ps": "surprised", "sad": "sad"
}

for root, _, files in os.walk(tess_dir):
    for f in files:
        if f.lower().endswith(".wav"):
            parts = f[:-4].split("_")
            code = parts[-1].lower()
            if code in tess_labels:
                records.append({
                    "path": os.path.join(root, f),
                    "dataset": "tess",
                    "emotion": tess_labels[code],
                    "speaker": "tess_" + parts[0]
                })

# RAVDESS — use the deduplicated paths from the previous cell
ravdess_labels = {
    "01": "neutral", "02": "calm", "03": "happy",
    "04": "sad", "05": "angry", "06": "fearful",
    "07": "disgust", "08": "surprised"
}

for path in unique_ravdess:
    f = os.path.basename(path)
    parts = f[:-4].split("-")

    if len(parts) == 7 and parts[0] == "03":
        emotion_code = parts[2]
        if emotion_code in ravdess_labels:
            records.append({
                "path": path,
                "dataset": "ravdess",
                "emotion": ravdess_labels[emotion_code],
                "speaker": "ravdess_" + parts[6]
            })

df = pd.DataFrame(records).drop_duplicates(subset="path").reset_index(drop=True)

print("Total recordings:", len(df))
print("\nBy dataset:")
print(df["dataset"].value_counts())
print("\nBy emotion:")
print(df["emotion"].value_counts())
print("\nUnique speakers:", df["speaker"].nunique())

Total recordings: 11681

By dataset:
dataset
crema      7442
tess       2800
ravdess    1439
Name: count, dtype: int64

By emotion:
emotion
sad          1863
fearful      1863
disgust      1863
angry        1863
happy        1862
neutral      1583
surprised     592
calm          192
Name: count, dtype: int64

Unique speakers: 117


In [18]:
from sklearn.model_selection import GroupShuffleSplit

# Training and validation data
trainval_df = df[df["dataset"].isin(["crema", "ravdess"])].copy()

# Keep TESS separate for external testing
tess_test_df = df[df["dataset"] == "tess"].copy()

# Split by speaker to reduce speaker leakage
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(trainval_df, groups=trainval_df["speaker"])
)

train_df = trainval_df.iloc[train_idx].reset_index(drop=True)
val_df = trainval_df.iloc[val_idx].reset_index(drop=True)
tess_test_df = tess_test_df.reset_index(drop=True)

print("Training recordings:", len(train_df))
print("Validation recordings:", len(val_df))
print("TESS test recordings:", len(tess_test_df))

train_speakers = set(train_df["speaker"])
val_speakers = set(val_df["speaker"])

print("Speakers shared between train and validation:",
      len(train_speakers & val_speakers))
print("Training emotion labels:", sorted(train_df["emotion"].unique()))
print("Validation emotion labels:", sorted(val_df["emotion"].unique()))

Training recordings: 7111
Validation recordings: 1770
TESS test recordings: 2800
Speakers shared between train and validation: 0
Training emotion labels: ['angry', 'calm', 'disgust', 'fearful', 'happy', 'neutral', 'sad', 'surprised']
Validation emotion labels: ['angry', 'calm', 'disgust', 'fearful', 'happy', 'neutral', 'sad', 'surprised']


In [19]:
!pip install -q transformers datasets accelerate soundfile librosa scikit-learn


In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU — training will be slower.")

PyTorch version: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

project_dir = "/content/drive/MyDrive/SpeechEmotionProject"

print("Project files:", os.listdir(project_dir))
print(
    "RAVDESS archive exists:",
    os.path.exists(os.path.join(project_dir, "ravdess.zip"))
)

Project files: ['ravdess.zip', 'checkpoints', 'final_model', 'experiment_2', 'final_model_v2']
RAVDESS archive exists: True


In [4]:
import os
import zipfile

# Restore RAVDESS from Google Drive if it isn't extracted
ravdess_dir = "/content/ravdess_extracted"
ravdess_zip = "/content/drive/MyDrive/SpeechEmotionProject/ravdess.zip"

if not os.path.isdir(ravdess_dir):
    os.makedirs(ravdess_dir, exist_ok=True)
    with zipfile.ZipFile(ravdess_zip, "r") as z:
        z.extractall(ravdess_dir)

# Check whether CREMA-D and TESS still exist
print("RAVDESS WAV files:",
      sum(f.lower().endswith(".wav")
          for root, _, files in os.walk(ravdess_dir)
          for f in files))

print("CREMA-D exists:", os.path.isdir("/content/CREMA-D/AudioWAV"))
print("TESS exists:", os.path.isdir("/content/tess_extracted"))

RAVDESS WAV files: 2880
CREMA-D exists: False
TESS exists: False


In [5]:
import os
import requests
import zipfile

# CREMA-D
crema_dir = "/content/CREMA-D/AudioWAV"

if not os.path.isdir(crema_dir):
    os.makedirs("/content", exist_ok=True)
    url = "https://github.com/CheyneyComputerScience/CREMA-D/archive/refs/heads/master.zip"
    response = requests.get(url, timeout=180)
    response.raise_for_status()

    with open("/content/crema.zip", "wb") as f:
        f.write(response.content)

    with zipfile.ZipFile("/content/crema.zip") as z:
        z.extractall("/content/crema_extract")

    # Locate the AudioWAV folder
    for root, dirs, files in os.walk("/content/crema_extract"):
        if os.path.basename(root) == "AudioWAV":
            crema_dir = root
            break

# TESS
tess_dir = "/content/tess_extracted"

if not os.path.isdir(tess_dir):
    url = (
        "https://borealisdata.ca/api/access/dataset/:persistentId"
        "?persistentId=doi:10.5683/SP2/E8H2MF"
    )
    response = requests.get(url, timeout=180)
    response.raise_for_status()

    with open("/content/tess.zip", "wb") as f:
        f.write(response.content)

    with zipfile.ZipFile("/content/tess.zip") as z:
        z.extractall(tess_dir)

print("CREMA-D folder:", crema_dir)
print("CREMA-D WAVs:", sum(
    f.lower().endswith(".wav")
    for _, _, files in os.walk(crema_dir) for f in files
))
print("TESS WAVs:", sum(
    f.lower().endswith(".wav")
    for _, _, files in os.walk(tess_dir) for f in files
))

CREMA-D folder: /content/crema_extract/CREMA-D-master/AudioWAV
CREMA-D WAVs: 7442
TESS WAVs: 2800


In [6]:
import os
import hashlib
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

records = []

# 1. CREMA-D
crema_dir = "/content/crema_extract/CREMA-D-master/AudioWAV"
crema_labels = {
    "ANG": "angry", "DIS": "disgust", "FEA": "fearful",
    "HAP": "happy", "NEU": "neutral", "SAD": "sad"
}

for f in os.listdir(crema_dir):
    if f.lower().endswith(".wav"):
        parts = f.split("_")
        if len(parts) >= 3 and parts[2] in crema_labels:
            records.append({
                "path": os.path.join(crema_dir, f),
                "dataset": "crema",
                "emotion": crema_labels[parts[2]],
                "speaker": "crema_" + parts[0]
            })

# 2. TESS
tess_dir = "/content/tess_extracted"
tess_labels = {
    "angry": "angry", "disgust": "disgust", "fear": "fearful",
    "happy": "happy", "neutral": "neutral",
    "ps": "surprised", "sad": "sad"
}

for root, _, files in os.walk(tess_dir):
    for f in files:
        if f.lower().endswith(".wav"):
            parts = f[:-4].split("_")
            code = parts[-1].lower()
            if code in tess_labels:
                records.append({
                    "path": os.path.join(root, f),
                    "dataset": "tess",
                    "emotion": tess_labels[code],
                    "speaker": "tess_" + parts[0]
                })

# 3. RAVDESS — deduplicate by audio file hash
ravdess_dir = "/content/ravdess_extracted"
seen_hashes = set()
ravdess_unique = []

for root, _, files in os.walk(ravdess_dir):
    for f in sorted(files):
        if f.lower().endswith(".wav"):
            path = os.path.join(root, f)
            h = hashlib.sha256()
            with open(path, "rb") as audio_file:
                for chunk in iter(lambda: audio_file.read(1024 * 1024), b""):
                    h.update(chunk)
            digest = h.hexdigest()

            if digest not in seen_hashes:
                seen_hashes.add(digest)
                ravdess_unique.append(path)

ravdess_labels = {
    "01": "neutral", "02": "calm", "03": "happy",
    "04": "sad", "05": "angry", "06": "fearful",
    "07": "disgust", "08": "surprised"
}

for path in ravdess_unique:
    f = os.path.basename(path)
    parts = f[:-4].split("-")
    if len(parts) == 7 and parts[0] == "03":
        if parts[2] in ravdess_labels:
            records.append({
                "path": path,
                "dataset": "ravdess",
                "emotion": ravdess_labels[parts[2]],
                "speaker": "ravdess_" + parts[6]
            })

# 4. Create dataset index
df = pd.DataFrame(records).drop_duplicates(subset="path").reset_index(drop=True)

# 5. Split by speaker; hold TESS out for external testing
trainval_df = df[df["dataset"].isin(["crema", "ravdess"])].copy()
tess_test_df = df[df["dataset"] == "tess"].copy()

splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)
train_idx, val_idx = next(
    splitter.split(trainval_df, groups=trainval_df["speaker"])
)

train_df = trainval_df.iloc[train_idx].reset_index(drop=True)
val_df = trainval_df.iloc[val_idx].reset_index(drop=True)
tess_test_df = tess_test_df.reset_index(drop=True)

print("Total recordings:", len(df))
print(df["dataset"].value_counts())
print("\nTrain:", len(train_df))
print("Validation:", len(val_df))
print("TESS test:", len(tess_test_df))
print("Shared train/validation speakers:",
      len(set(train_df["speaker"]) & set(val_df["speaker"])))

Total recordings: 11681
dataset
crema      7442
tess       2800
ravdess    1439
Name: count, dtype: int64

Train: 7111
Validation: 1770
TESS test: 2800
Shared train/validation speakers: 0


In [7]:
from transformers import AutoFeatureExtractor

MODEL_NAME = "facebook/wav2vec2-base"

emotion_labels = [
    "angry", "calm", "disgust", "fearful",
    "happy", "neutral", "sad", "surprised"
]

label2id = {label: i for i, label in enumerate(emotion_labels)}
id2label = {i: label for label, i in label2id.items()}

train_df["labels"] = train_df["emotion"].map(label2id)
val_df["labels"] = val_df["emotion"].map(label2id)
tess_test_df["labels"] = tess_test_df["emotion"].map(label2id)

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

print("Feature extractor loaded.")
print("Labels:", label2id)
print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))
print("TESS test rows:", len(tess_test_df))

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

Feature extractor loaded.
Labels: {'angry': 0, 'calm': 1, 'disgust': 2, 'fearful': 3, 'happy': 4, 'neutral': 5, 'sad': 6, 'surprised': 7}
Training rows: 7111
Validation rows: 1770
TESS test rows: 2800


In [8]:
from datasets import Dataset, Audio

# Convert the dataframes into Hugging Face datasets
train_ds = Dataset.from_pandas(
    train_df[["path", "labels"]], preserve_index=False
).rename_column("path", "audio")

val_ds = Dataset.from_pandas(
    val_df[["path", "labels"]], preserve_index=False
).rename_column("path", "audio")

tess_ds = Dataset.from_pandas(
    tess_test_df[["path", "labels"]], preserve_index=False
).rename_column("path", "audio")

# Decode audio and resample to 16 kHz
train_ds = train_ds.cast_column(
    "audio", Audio(sampling_rate=16000)
)
val_ds = val_ds.cast_column(
    "audio", Audio(sampling_rate=16000)
)
tess_ds = tess_ds.cast_column(
    "audio", Audio(sampling_rate=16000)
)

print("Datasets created:")
print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("TESS test:", len(tess_ds))


Datasets created:
Train: 7111
Validation: 1770
TESS test: 2800


In [9]:
import numpy as np

MAX_AUDIO_SECONDS = 6

def preprocess_batch(batch):
    audio_arrays = [item["array"] for item in batch["audio"]]

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        truncation=True,
        max_length=16000 * MAX_AUDIO_SECONDS,
        padding=False,
    )

    inputs["labels"] = batch["labels"]
    return inputs

train_ready = train_ds.map(
    preprocess_batch,
    batched=True,
    batch_size=16,
    remove_columns=train_ds.column_names,
    desc="Preprocessing training audio",
)

val_ready = val_ds.map(
    preprocess_batch,
    batched=True,
    batch_size=16,
    remove_columns=val_ds.column_names,
    desc="Preprocessing validation audio",
)

tess_ready = tess_ds.map(
    preprocess_batch,
    batched=True,
    batch_size=16,
    remove_columns=tess_ds.column_names,
    desc="Preprocessing TESS audio",
)

print("Preprocessing complete!")
print("Train:", len(train_ready))
print("Validation:", len(val_ready))
print("TESS test:", len(tess_ready))
print("Input keys:", train_ready[0].keys())

Preprocessing training audio:   0%|          | 0/7111 [00:00<?, ? examples/s]

RuntimeError: SingleStreamDecoder, /__w/torchcodec/torchcodec/meta-pytorch/torchcodec/src/torchcodec/_core/SingleStreamDecoder.cpp:51, Could not open input file: /content/crema_extract/CREMA-D-master/AudioWAV/1029_MTI_NEU_XX.wav Invalid data found when processing input

In [10]:
import soundfile as sf

bad_path = "/content/crema_extract/CREMA-D-master/AudioWAV/1029_MTI_NEU_XX.wav"

try:
    audio, sr = sf.read(bad_path)
    print("File readable with soundfile")
    print("Shape:", audio.shape)
    print("Sample rate:", sr)
except Exception as e:
    print("File could not be decoded:", repr(e))

File could not be decoded: LibsndfileError(1, "Error opening '/content/crema_extract/CREMA-D-master/AudioWAV/1029_MTI_NEU_XX.wav': ")


In [11]:
import os
import numpy as np
import soundfile as sf
import librosa
from datasets import Dataset

MAX_AUDIO_SECONDS = 6

# Keep only files that soundfile can actually read.
def readable(path):
    try:
        info = sf.info(path)
        return info.frames > 0 and info.samplerate > 0
    except Exception:
        return False

for name in ["train_df", "val_df", "tess_test_df"]:
    frame = globals()[name]
    valid = frame["path"].apply(readable)
    print(f"{name}: skipping {(~valid).sum()} unreadable files")
    globals()[name] = frame[valid].reset_index(drop=True)

# Build datasets from file paths, avoiding the decoder that failed earlier.
def make_dataset(frame):
    return Dataset.from_pandas(
        frame[["path", "labels"]], preserve_index=False
    )

train_ds = make_dataset(train_df)
val_ds = make_dataset(val_df)
tess_ds = make_dataset(tess_test_df)

def preprocess_batch(batch):
    audio_arrays = []

    for path in batch["path"]:
        audio, sr = sf.read(path, dtype="float32", always_2d=False)

        # Convert stereo recordings to mono.
        if audio.ndim == 2:
            audio = audio.mean(axis=1)

        if sr != 16000:
            audio = librosa.resample(
                audio, orig_sr=sr, target_sr=16000
            )

        audio = audio[:16000 * MAX_AUDIO_SECONDS]
        audio_arrays.append(audio)

    result = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        padding=False,
        truncation=True,
        max_length=16000 * MAX_AUDIO_SECONDS,
    )
    result["labels"] = batch["labels"]
    return result

def process_dataset(ds, name):
    return ds.map(
        preprocess_batch,
        batched=True,
        batch_size=8,
        remove_columns=ds.column_names,
        desc=f"Preprocessing {name}",
    )

train_ready = process_dataset(train_ds, "training")
val_ready = process_dataset(val_ds, "validation")
tess_ready = process_dataset(tess_ds, "TESS test")

print("\nPreprocessing finished.")
print("Train:", len(train_ready))
print("Validation:", len(val_ready))
print("TESS test:", len(tess_ready))
print("Input keys:", train_ready[0].keys())

train_df: skipping 5972 unreadable files
val_df: skipping 1470 unreadable files
tess_test_df: skipping 0 unreadable files


Preprocessing training:   0%|          | 0/1139 [00:00<?, ? examples/s]

Preprocessing validation:   0%|          | 0/300 [00:00<?, ? examples/s]

Preprocessing TESS test:   0%|          | 0/2800 [00:00<?, ? examples/s]


Preprocessing finished.
Train: 1139
Validation: 300
TESS test: 2800
Input keys: dict_keys(['labels', 'input_values'])


In [12]:
import os
import soundfile as sf

folder = "/content/crema_extract/CREMA-D-master/AudioWAV"

for name in os.listdir(folder)[:10]:
    path = os.path.join(folder, name)
    print(name, "—", os.path.getsize(path), "bytes")
    try:
        print("  Soundfile:", sf.info(path))
    except Exception as e:
        print("  Soundfile error:", str(e)[:120])

1029_MTI_NEU_XX.wav — 130 bytes
  Soundfile error: Error opening '/content/crema_extract/CREMA-D-master/AudioWAV/1029_MTI_NEU_XX.wav': Format not recognised.
1011_IOM_DIS_XX.wav — 130 bytes
  Soundfile error: Error opening '/content/crema_extract/CREMA-D-master/AudioWAV/1011_IOM_DIS_XX.wav': Format not recognised.
1070_TIE_SAD_XX.wav — 131 bytes
  Soundfile error: Error opening '/content/crema_extract/CREMA-D-master/AudioWAV/1070_TIE_SAD_XX.wav': Format not recognised.
1008_IWL_ANG_XX.wav — 131 bytes
  Soundfile error: Error opening '/content/crema_extract/CREMA-D-master/AudioWAV/1008_IWL_ANG_XX.wav': Format not recognised.
1034_TSI_HAP_XX.wav — 130 bytes
  Soundfile error: Error opening '/content/crema_extract/CREMA-D-master/AudioWAV/1034_TSI_HAP_XX.wav': Format not recognised.
1045_IEO_SAD_LO.wav — 130 bytes
  Soundfile error: Error opening '/content/crema_extract/CREMA-D-master/AudioWAV/1045_IEO_SAD_LO.wav': Format not recognised.
1017_IEO_FEA_MD.wav — 130 bytes
  Soundfile error: E

In [13]:
!pip -q install huggingface_hub

from huggingface_hub import hf_hub_download
import os
import tarfile

archive = hf_hub_download(
    repo_id="myleslinder/crema-d",
    filename="data/crema_d.tar.gz",
    repo_type="dataset"
)

extract_dir = "/content/crema_real"
os.makedirs(extract_dir, exist_ok=True)

with tarfile.open(archive, "r:gz") as tar:
    tar.extractall(extract_dir, filter="data")

# Find actual WAV files
real_wavs = []
for root, _, files in os.walk(extract_dir):
    for f in files:
        if f.lower().endswith(".wav"):
            path = os.path.join(root, f)
            if os.path.getsize(path) > 1000:
                real_wavs.append(path)

print("Real WAV files found:", len(real_wavs))
print("Example:", real_wavs[0] if real_wavs else "None found")

data/crema_d.tar.gz: reconstructing file:   0%|          |  0.00B /  471MB            

data/crema_d.tar.gz: downloading bytes:           |  0.00B            

Real WAV files found: 7442
Example: /content/crema_real/data/AudioWAV/1029_MTI_NEU_XX.wav


In [14]:
import os
import soundfile as sf

crema_dir = "/content/crema_real/data/AudioWAV"
ravdess_dir = "/content/ravdess_extracted"
tess_dir = "/content/tess_extracted"

def check_audio(folder, limit=20):
    files = []
    for root, _, names in os.walk(folder):
        for name in names:
            if name.lower().endswith(".wav"):
                files.append(os.path.join(root, name))

    readable = 0
    checked = 0

    for path in files[:limit]:
        try:
            sf.info(path)
            readable += 1
        except Exception:
            pass
        checked += 1

    print(f"\n{folder}")
    print(f"WAV files found: {len(files)}")
    print(f"Readable in sample: {readable}/{checked}")

check_audio(crema_dir)
check_audio(ravdess_dir)
check_audio(tess_dir)


/content/crema_real/data/AudioWAV
WAV files found: 7442
Readable in sample: 20/20

/content/ravdess_extracted
WAV files found: 2880
Readable in sample: 20/20

/content/tess_extracted
WAV files found: 2800
Readable in sample: 20/20


In [15]:
import os
import hashlib
import pandas as pd
import soundfile as sf

DATASETS = {
    "CREMA-D": "/content/crema_real/data/AudioWAV",
    "RAVDESS": "/content/ravdess_extracted",
    "TESS": "/content/tess_extracted",
}

emotion_labels = [
    "angry", "calm", "disgust", "fearful",
    "happy", "neutral", "sad", "surprised"
]
label2id = {label: i for i, label in enumerate(emotion_labels)}
records = []

# CREMA-D filename example: 1029_MTI_NEU_XX.wav
crema_map = {
    "ANG": "angry", "DIS": "disgust", "FEA": "fearful",
    "HAP": "happy", "NEU": "neutral", "SAD": "sad"
}

# RAVDESS emotion codes
ravdess_map = {
    "01": "neutral", "02": "calm", "03": "happy",
    "04": "sad", "05": "angry", "06": "fearful",
    "07": "disgust", "08": "surprised"
}

# TESS emotion names
tess_map = {
    "angry": "angry", "disgust": "disgust",
    "fear": "fearful", "fearful": "fearful",
    "happy": "happy", "neutral": "neutral",
    "sad": "sad", "surprise": "surprised",
    "surprised": "surprised"
}

seen_hashes = set()
skipped = 0

for dataset, folder in DATASETS.items():
    for root, _, files in os.walk(folder):
        for filename in files:
            if not filename.lower().endswith(".wav"):
                continue

            path = os.path.join(root, filename)
            name = filename.lower()

            try:
                sf.info(path)
                with open(path, "rb") as f:
                    digest = hashlib.sha256(f.read()).hexdigest()
            except Exception:
                skipped += 1
                continue

            if digest in seen_hashes:
                continue

            emotion = None

            if dataset == "CREMA-D":
                parts = filename.split("_")
                if len(parts) >= 3:
                    emotion = crema_map.get(parts[2].upper())

            elif dataset == "RAVDESS":
                parts = filename.split("-")
                if len(parts) >= 7:
                    emotion = ravdess_map.get(parts[2])

            elif dataset == "TESS":
                final_word = name[:-4].split("_")[-1]
                emotion = tess_map.get(final_word)

            if emotion not in label2id:
                continue

            seen_hashes.add(digest)
            records.append({
                "path": path,
                "emotion": emotion,
                "label": label2id[emotion],
                "dataset": dataset
            })

df = pd.DataFrame(records)

print("Total unique recordings:", len(df))
print("\nDataset counts:")
print(df["dataset"].value_counts())
print("\nEmotion counts:")
print(df["emotion"].value_counts())
print("\nUnreadable files skipped:", skipped)
print("\nSample:")
display(df.head())

Total unique recordings: 11278

Dataset counts:
dataset
CREMA-D    7439
TESS       2400
RAVDESS    1439
Name: count, dtype: int64

Emotion counts:
emotion
disgust      1863
angry        1863
sad          1862
fearful      1862
happy        1861
neutral      1583
surprised     192
calm          192
Name: count, dtype: int64

Unreadable files skipped: 0

Sample:


,path,emotion,label,dataset
0,/content/crema_real/data/AudioWAV/1029_MTI_NEU...,neutral,5,CREMA-D
1,/content/crema_real/data/AudioWAV/1011_IOM_DIS...,disgust,2,CREMA-D
2,/content/crema_real/data/AudioWAV/1070_TIE_SAD...,sad,6,CREMA-D
3,/content/crema_real/data/AudioWAV/1008_IWL_ANG...,angry,0,CREMA-D
4,/content/crema_real/data/AudioWAV/1034_TSI_HAP...,happy,4,CREMA-D


In [16]:
from sklearn.model_selection import GroupShuffleSplit

# TESS stays separate for external testing
tess_df = df[df["dataset"] == "TESS"].copy()

# Train and validation use CREMA-D + RAVDESS
development_df = df[df["dataset"] != "TESS"].copy()

# Extract speaker IDs from filenames
def get_speaker_id(row):
    filename = os.path.basename(row["path"])

    if row["dataset"] == "CREMA-D":
        return "CREMA_" + filename.split("_")[0]

    if row["dataset"] == "RAVDESS":
        parts = filename.split("-")
        return "RAVDESS_" + parts[6].replace(".wav", "")

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

groups = development_df.apply(get_speaker_id, axis=1)

train_idx, val_idx = next(
    splitter.split(development_df, groups=groups)
)

train_df = development_df.iloc[train_idx].reset_index(drop=True)
val_df = development_df.iloc[val_idx].reset_index(drop=True)
tess_df = tess_df.reset_index(drop=True)

print("Training recordings:", len(train_df))
print("Validation recordings:", len(val_df))
print("External TESS test recordings:", len(tess_df))

print("\nTraining labels:")
print(train_df["emotion"].value_counts().sort_index())

print("\nValidation labels:")
print(val_df["emotion"].value_counts().sort_index())

train_speakers = set(groups.iloc[train_idx])
val_speakers = set(groups.iloc[val_idx])
print("\nSpeakers shared between train and validation:",
      len(train_speakers & val_speakers))

Training recordings: 7109
Validation recordings: 1769
External TESS test recordings: 2400

Training labels:
emotion
angry        1172
calm          152
disgust      1172
fearful      1171
happy        1170
neutral       948
sad          1172
surprised     152
Name: count, dtype: int64

Validation labels:
emotion
angry        291
calm          40
disgust      291
fearful      291
happy        291
neutral      235
sad          290
surprised     40
Name: count, dtype: int64

Speakers shared between train and validation: 0


In [ ]:
import numpy as np
import soundfile as sf
import librosa
from datasets import Dataset
from transformers import AutoFeatureExtractor
from tqdm.auto import tqdm

MODEL_NAME = "facebook/wav2vec2-base"
SAMPLE_RATE = 16000
MAX_SECONDS = 6

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

def preprocess_dataframe(frame, name):
    processed = []
    skipped = 0

    for row in tqdm(frame.itertuples(index=False), total=len(frame), desc=name):
        try:
            audio, sr = sf.read(
                row.path,
                dtype="float32",
                always_2d=False
            )

            # Convert stereo audio to mono
            if audio.ndim == 2:
                audio = audio.mean(axis=1)

            # Resample to 16 kHz when needed
            if sr != SAMPLE_RATE:
                audio = librosa.resample(
                    audio,
                    orig_sr=sr,
                    target_sr=SAMPLE_RATE
                )

            # Avoid empty or invalid recordings
            if len(audio) == 0 or not np.isfinite(audio).all():
                skipped += 1
                continue

            # Keep at most 6 seconds
            audio = audio[:SAMPLE_RATE * MAX_SECONDS]

            features = feature_extractor(
                audio,
                sampling_rate=SAMPLE_RATE,
                truncation=True,
                max_length=SAMPLE_RATE * MAX_SECONDS
            )

            processed.append({
                "input_values": features["input_values"][0],
                "labels": int(row.label)
            })

        except Exception as e:
            skipped += 1

    print(f"{name}: processed {len(processed)}, skipped {skipped}")
    return Dataset.from_list(processed)

train_dataset = preprocess_dataframe(train_df, "Training")
val_dataset = preprocess_dataframe(val_df, "Validation")
tess_dataset = preprocess_dataframe(tess_df, "TESS test")

print("\nFinal dataset sizes:")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("TESS test:", len(tess_dataset))
print("Feature keys:", train_dataset.column_names)

Training:   0%|          | 0/7109 [00:00<?, ?it/s]

Training: processed 7109, skipped 0


In [1]:
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("TESS test:", len(tess_dataset))
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

NameError: name 'train_dataset' is not defined

In [2]:
print("train_df exists:", "train_df" in globals())
print("val_df exists:", "val_df" in globals())
print("tess_df exists:", "tess_df" in globals())
print("preprocess_dataframe exists:", "preprocess_dataframe" in globals())
print("train_dataset exists:", "train_dataset" in globals())

train_df exists: False
val_df exists: False
tess_df exists: False
preprocess_dataframe exists: False
train_dataset exists: False


In [3]:
import os
import hashlib
import pandas as pd
import soundfile as sf
from sklearn.model_selection import GroupShuffleSplit

DATASETS = {
    "CREMA-D": "/content/crema_real/data/AudioWAV",
    "RAVDESS": "/content/ravdess_extracted",
    "TESS": "/content/tess_extracted",
}

emotion_labels = [
    "angry", "calm", "disgust", "fearful",
    "happy", "neutral", "sad", "surprised"
]
label2id = {label: i for i, label in enumerate(emotion_labels)}

crema_map = {
    "ANG": "angry", "DIS": "disgust", "FEA": "fearful",
    "HAP": "happy", "NEU": "neutral", "SAD": "sad"
}
ravdess_map = {
    "01": "neutral", "02": "calm", "03": "happy",
    "04": "sad", "05": "angry", "06": "fearful",
    "07": "disgust", "08": "surprised"
}
tess_map = {
    "angry": "angry", "disgust": "disgust",
    "fear": "fearful", "fearful": "fearful",
    "happy": "happy", "neutral": "neutral",
    "sad": "sad", "surprise": "surprised",
    "surprised": "surprised"
}

records = []
seen_hashes = set()

for dataset, folder in DATASETS.items():
    if not os.path.isdir(folder):
        raise FileNotFoundError(
            f"Missing folder: {folder}. The runtime may have reset."
        )

    for root, _, files in os.walk(folder):
        for filename in files:
            if not filename.lower().endswith(".wav"):
                continue

            path = os.path.join(root, filename)

            try:
                sf.info(path)
                with open(path, "rb") as f:
                    digest = hashlib.sha256(f.read()).hexdigest()
            except Exception:
                continue

            if digest in seen_hashes:
                continue

            if dataset == "CREMA-D":
                parts = filename.split("_")
                emotion = crema_map.get(parts[2].upper()) if len(parts) >= 3 else None
                speaker = "CREMA_" + parts[0]

            elif dataset == "RAVDESS":
                parts = filename.split("-")
                emotion = ravdess_map.get(parts[2]) if len(parts) >= 7 else None
                speaker = "RAVDESS_" + parts[6].replace(".wav", "") if len(parts) >= 7 else None

            else:
                emotion = tess_map.get(filename.lower()[:-4].split("_")[-1])
                speaker = "TESS_" + filename.split("_")[0]

            if emotion not in label2id:
                continue

            seen_hashes.add(digest)
            records.append({
                "path": path,
                "emotion": emotion,
                "label": label2id[emotion],
                "dataset": dataset,
                "speaker": speaker
            })

df = pd.DataFrame(records)

tess_df = df[df["dataset"] == "TESS"].copy().reset_index(drop=True)
development_df = df[df["dataset"] != "TESS"].copy().reset_index(drop=True)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, val_idx = next(
    splitter.split(development_df, groups=development_df["speaker"])
)

train_df = development_df.iloc[train_idx].reset_index(drop=True)
val_df = development_df.iloc[val_idx].reset_index(drop=True)

print("Total unique recordings:", len(df))
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("TESS test:", len(tess_df))
print(
    "Speakers shared:",
    len(set(train_df["speaker"]) & set(val_df["speaker"]))
)

Total unique recordings: 11278
Train: 7109
Validation: 1769
TESS test: 2400
Speakers shared: 0


In [1]:
import os
import numpy as np
import soundfile as sf
import librosa
import torch
from datasets import Dataset, DatasetDict, load_from_disk
from transformers import AutoFeatureExtractor
from tqdm.auto import tqdm

MODEL_NAME = "facebook/wav2vec2-base"
SAMPLE_RATE = 16000
MAX_SECONDS = 6

save_dir = "/content/drive/MyDrive/SpeechEmotionProject/processed_datasets"

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

def preprocess_dataframe(frame, name):
    processed = []
    skipped = 0

    for row in tqdm(frame.itertuples(index=False), total=len(frame), desc=name):
        try:
            audio, sr = sf.read(
                row.path, dtype="float32", always_2d=False
            )

            if audio.ndim == 2:
                audio = audio.mean(axis=1)

            if sr != SAMPLE_RATE:
                audio = librosa.resample(
                    audio, orig_sr=sr, target_sr=SAMPLE_RATE
                )

            if len(audio) == 0 or not np.isfinite(audio).all():
                skipped += 1
                continue

            audio = audio[:SAMPLE_RATE * MAX_SECONDS]

            features = feature_extractor(
                audio,
                sampling_rate=SAMPLE_RATE,
                truncation=True,
                max_length=SAMPLE_RATE * MAX_SECONDS
            )

            processed.append({
                "input_values": features["input_values"][0],
                "labels": int(row.label)
            })

        except Exception as e:
            skipped += 1

    print(f"{name}: processed {len(processed)}, skipped {skipped}")
    return Dataset.from_list(processed)

train_dataset = preprocess_dataframe(train_df, "Training")
val_dataset = preprocess_dataframe(val_df, "Validation")
tess_dataset = preprocess_dataframe(tess_df, "TESS test")

datasets = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": tess_dataset
})

datasets.save_to_disk(save_dir)

print("\nSaved datasets to:", save_dir)
print(datasets)

NameError: name 'train_df' is not defined

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os

save_dir = "/content/drive/MyDrive/SpeechEmotionProject/processed_datasets"

print("Saved datasets exist:", os.path.isdir(save_dir))
if os.path.isdir(save_dir):
    print("Contents:", os.listdir(save_dir))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved datasets exist: False


In [ ]:
import os, hashlib
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit
from transformers import AutoFeatureExtractor
from datasets import Dataset, DatasetDict

# Dataset locations
folders = {
    "CREMA-D": "/content/crema_real/data/AudioWAV",
    "RAVDESS": "/content/ravdess_extracted",
    "TESS": "/content/tess_extracted"
}

labels = ["angry", "calm", "disgust", "fearful",
          "happy", "neutral", "sad", "surprised"]
label2id = {x: i for i, x in enumerate(labels)}

crema_map = {"ANG":"angry","DIS":"disgust","FEA":"fearful",
             "HAP":"happy","NEU":"neutral","SAD":"sad"}
ravdess_map = {"01":"neutral","02":"calm","03":"happy","04":"sad",
               "05":"angry","06":"fearful","07":"disgust","08":"surprised"}
tess_map = {"angry":"angry","disgust":"disgust","fear":"fearful",
            "happy":"happy","neutral":"neutral","sad":"sad",
            "surprise":"surprised"}

records, seen = [], set()

for dataset, folder in folders.items():
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Missing {folder}. Extracted files are unavailable.")

    for root, _, files in os.walk(folder):
        for fn in files:
            if not fn.lower().endswith(".wav"):
                continue
            path = os.path.join(root, fn)
            try:
                sf.info(path)
                with open(path, "rb") as f:
                    digest = hashlib.sha256(f.read()).hexdigest()
            except Exception:
                continue
            if digest in seen:
                continue

            parts = fn.split("-") if dataset == "RAVDESS" else fn.split("_")
            if dataset == "CREMA-D":
                emotion = crema_map.get(parts[2].upper()) if len(parts) >= 3 else None
                speaker = "CREMA_" + parts[0]
            elif dataset == "RAVDESS":
                emotion = ravdess_map.get(parts[2]) if len(parts) >= 7 else None
                speaker = "RAVDESS_" + parts[6].replace(".wav", "") if len(parts) >= 7 else None
            else:
                emotion = tess_map.get(fn.lower()[:-4].split("_")[-1])
                speaker = "TESS_" + parts[0]

            if emotion not in label2id:
                continue
            seen.add(digest)
            records.append({"path":path, "emotion":emotion,
                            "label":label2id[emotion],
                            "dataset":dataset, "speaker":speaker})

df = pd.DataFrame(records)
tess_df = df[df.dataset == "TESS"].copy().reset_index(drop=True)
dev = df[df.dataset != "TESS"].copy().reset_index(drop=True)
split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr, va = next(split.split(dev, groups=dev.speaker))
train_df = dev.iloc[tr].reset_index(drop=True)
val_df = dev.iloc[va].reset_index(drop=True)

print("Indexed:", len(df), "| Train:", len(train_df),
      "| Validation:", len(val_df), "| TESS:", len(tess_df))

# Preprocess
extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")

def prepare(frame, name):
    rows = []
    for r in tqdm(frame.itertuples(index=False), total=len(frame), desc=name):
        try:
            audio, sr = sf.read(r.path, dtype="float32", always_2d=False)
            if audio.ndim == 2:
                audio = audio.mean(axis=1)
            if sr != 16000:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
            audio = audio[:96000]
            if len(audio) == 0 or not np.isfinite(audio).all():
                continue
            feat = extractor(audio, sampling_rate=16000,
                             truncation=True, max_length=96000)
            rows.append({"input_values":feat["input_values"][0],
                         "labels":int(r.label)})
        except Exception:
            continue
    print(name, "processed:", len(rows))
    return Dataset.from_list(rows)

datasets = DatasetDict({
    "train": prepare(train_df, "Train"),
    "validation": prepare(val_df, "Validation"),
    "test": prepare(tess_df, "TESS")
})

save_dir = "/content/drive/MyDrive/SpeechEmotionProject/processed_datasets"
datasets.save_to_disk(save_dir)
print("SAVED:", save_dir)
print({k:len(v) for k,v in datasets.items()})

Indexed: 11278 | Train: 7109 | Validation: 1769 | TESS: 2400


Train:   0%|          | 0/7109 [00:00<?, ?it/s]

Train processed: 7109


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os

print("CREMA-D extracted:",
      os.path.isdir("/content/crema_real/data/AudioWAV"))
print("RAVDESS extracted:",
      os.path.isdir("/content/ravdess_extracted"))
print("TESS extracted:",
      os.path.isdir("/content/tess_extracted"))
print("Drive connected:",
      os.path.isdir("/content/drive/MyDrive/SpeechEmotionProject"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CREMA-D extracted: True
RAVDESS extracted: True
TESS extracted: True
Drive connected: True


In [2]:
import os, hashlib, json, gc
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch

from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit
from transformers import AutoFeatureExtractor

# ---------- Paths and labels ----------
folders = {
    "CREMA-D": "/content/crema_real/data/AudioWAV",
    "RAVDESS": "/content/ravdess_extracted",
    "TESS": "/content/tess_extracted"
}
save_root = "/content/drive/MyDrive/SpeechEmotionProject/processed_batches"
os.makedirs(save_root, exist_ok=True)

labels = ["angry", "calm", "disgust", "fearful",
          "happy", "neutral", "sad", "surprised"]
label2id = {x: i for i, x in enumerate(labels)}

crema_map = {
    "ANG":"angry", "DIS":"disgust", "FEA":"fearful",
    "HAP":"happy", "NEU":"neutral", "SAD":"sad"
}
ravdess_map = {
    "01":"neutral", "02":"calm", "03":"happy", "04":"sad",
    "05":"angry", "06":"fearful", "07":"disgust", "08":"surprised"
}
tess_map = {
    "angry":"angry", "disgust":"disgust", "fear":"fearful",
    "happy":"happy", "neutral":"neutral", "sad":"sad",
    "surprise":"surprised"
}

# ---------- Rebuild index ----------
records, seen = [], set()

for dataset, folder in folders.items():
    for root, _, files in os.walk(folder):
        for fn in files:
            if not fn.lower().endswith(".wav"):
                continue

            path = os.path.join(root, fn)
            try:
                sf.info(path)
                with open(path, "rb") as f:
                    digest = hashlib.sha256(f.read()).hexdigest()
            except Exception:
                continue

            if digest in seen:
                continue

            parts = fn.split("-") if dataset == "RAVDESS" else fn.split("_")

            if dataset == "CREMA-D":
                emotion = crema_map.get(parts[2].upper()) if len(parts) >= 3 else None
                speaker = "CREMA_" + parts[0]
            elif dataset == "RAVDESS":
                emotion = ravdess_map.get(parts[2]) if len(parts) >= 7 else None
                speaker = "RAVDESS_" + parts[6].replace(".wav", "") if len(parts) >= 7 else None
            else:
                emotion = tess_map.get(fn.lower()[:-4].split("_")[-1])
                speaker = "TESS_" + parts[0]

            if emotion not in label2id:
                continue

            seen.add(digest)
            records.append({
                "path": path, "emotion": emotion,
                "label": label2id[emotion],
                "dataset": dataset, "speaker": speaker
            })

df = pd.DataFrame(records)
tess_df = df[df.dataset == "TESS"].copy().reset_index(drop=True)
dev = df[df.dataset != "TESS"].copy().reset_index(drop=True)

splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)
tr, va = next(splitter.split(dev, groups=dev.speaker))
train_df = dev.iloc[tr].reset_index(drop=True)
val_df = dev.iloc[va].reset_index(drop=True)

print("Index:", len(df), "| Train:", len(train_df),
      "| Validation:", len(val_df), "| TESS:", len(tess_df))

# Save index so it can be recovered without re-indexing
df.to_csv(os.path.join(save_root, "index.csv"), index=False)
train_df.to_csv(os.path.join(save_root, "train.csv"), index=False)
val_df.to_csv(os.path.join(save_root, "validation.csv"), index=False)
tess_df.to_csv(os.path.join(save_root, "test.csv"), index=False)

# ---------- Feature extraction ----------
extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")
SAMPLE_RATE = 16000
MAX_SAMPLES = 96000
BATCH_SIZE = 64

def process_split(frame, split_name):
    split_dir = os.path.join(save_root, split_name)
    os.makedirs(split_dir, exist_ok=True)

    # Skip batches already saved, allowing safe reruns
    total_batches = (len(frame) + BATCH_SIZE - 1) // BATCH_SIZE
    completed = set()
    for fn in os.listdir(split_dir):
        if fn.startswith("batch_") and fn.endswith(".npz"):
            try:
                completed.add(int(fn[6:-4]))
            except ValueError:
                pass

    for batch_id in tqdm(range(total_batches), desc=split_name):
        if batch_id in completed:
            continue

        batch = frame.iloc[
            batch_id * BATCH_SIZE:(batch_id + 1) * BATCH_SIZE
        ]
        features_list, labels_list = [], []

        for row in batch.itertuples(index=False):
            try:
                audio, sr = sf.read(
                    row.path, dtype="float32", always_2d=False
                )
                if audio.ndim == 2:
                    audio = audio.mean(axis=1)
                if sr != SAMPLE_RATE:
                    audio = librosa.resample(
                        audio, orig_sr=sr, target_sr=SAMPLE_RATE
                    )
                audio = audio[:MAX_SAMPLES]

                if len(audio) == 0 or not np.isfinite(audio).all():
                    continue

                feature = extractor(
                    audio, sampling_rate=SAMPLE_RATE,
                    truncation=True, max_length=MAX_SAMPLES
                )
                features_list.append(
                    np.asarray(feature["input_values"][0], dtype=np.float32)
                )
                labels_list.append(int(row.label))
            except Exception as e:
                print("Skipping:", os.path.basename(row.path), str(e)[:100])

        # Each batch is saved immediately
        batch_path = os.path.join(split_dir, f"batch_{batch_id:04d}.npz")
        np.savez(
            batch_path,
            input_values=np.array(features_list, dtype=object),
            labels=np.asarray(labels_list, dtype=np.int64)
        )
        del features_list, labels_list
        gc.collect()

    print(split_name, "batch files:", len([
        f for f in os.listdir(split_dir)
        if f.endswith(".npz")
    ]))

process_split(train_df, "train")
process_split(val_df, "validation")
process_split(tess_df, "test")

print("\nAll batches saved under:", save_root)

Index: 11278 | Train: 7109 | Validation: 1769 | TESS: 2400


train:   0%|          | 0/112 [00:00<?, ?it/s]

train batch files: 112


validation:   0%|          | 0/28 [00:00<?, ?it/s]

validation batch files: 28


test:   0%|          | 0/38 [00:00<?, ?it/s]

test batch files: 38

All batches saved under: /content/drive/MyDrive/SpeechEmotionProject/processed_batches


In [ ]:
import os, gc, random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

# Reproducibility and GPU
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

root = "/content/drive/MyDrive/SpeechEmotionProject/processed_batches"
checkpoint_dir = "/content/drive/MyDrive/SpeechEmotionProject/combined_model_v1"
os.makedirs(checkpoint_dir, exist_ok=True)

emotion_labels = [
    "angry", "calm", "disgust", "fearful",
    "happy", "neutral", "sad", "surprised"
]
label2id = {name: i for i, name in enumerate(emotion_labels)}
id2label = {i: name for name, i in label2id.items()}

# Dataset reads batches from Drive rather than keeping all audio in RAM
class BatchAudioDataset(Dataset):
    def __init__(self, folder):
        self.files = sorted(
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.endswith(".npz")
        )
        self.index = []
        self.cache_file = None
        self.cache_data = None

        for file_id, path in enumerate(self.files):
            with np.load(path, allow_pickle=True) as data:
                self.index.extend(
                    (file_id, i) for i in range(len(data["labels"]))
                )

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        file_id, row_id = self.index[idx]
        path = self.files[file_id]

        if self.cache_file != path:
            self.cache_data = np.load(path, allow_pickle=True)
            self.cache_file = path

        x = np.asarray(
            self.cache_data["input_values"][row_id],
            dtype=np.float32
        ).copy()
        y = int(self.cache_data["labels"][row_id])

        return {"input_values": torch.from_numpy(x), "labels": y}

# Shuffle the batch files each epoch, keeping reads efficient
class ChunkSampler(Sampler):
    def __init__(self, dataset, chunk_size=64):
        self.dataset = dataset
        self.chunk_size = chunk_size

    def __iter__(self):
        chunks = [
            list(range(i, min(i + self.chunk_size, len(self.dataset))))
            for i in range(0, len(self.dataset), self.chunk_size)
        ]
        random.shuffle(chunks)
        for chunk in chunks:
            random.shuffle(chunk)
            yield from chunk

    def __len__(self):
        return len(self.dataset)

extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")

def collate_audio(items):
    inputs = [{"input_values": item["input_values"].numpy()} for item in items]
    batch = extractor.pad(inputs, padding=True, return_tensors="pt")
    batch["labels"] = torch.tensor(
        [item["labels"] for item in items], dtype=torch.long
    )
    return batch

train_ds = BatchAudioDataset(os.path.join(root, "train"))
val_ds = BatchAudioDataset(os.path.join(root, "validation"))

print("Loaded train examples:", len(train_ds))
print("Loaded validation examples:", len(val_ds))

train_loader = DataLoader(
    train_ds, batch_size=2, sampler=ChunkSampler(train_ds),
    collate_fn=collate_audio, num_workers=0
)
val_loader = DataLoader(
    val_ds, batch_size=2, shuffle=False,
    collate_fn=collate_audio, num_workers=0
)

model = AutoModelForAudioClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=8,
    label2id=label2id,
    id2label=id2label
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
EPOCHS = 3
ACCUMULATION_STEPS = 4
best_f1 = -1.0

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    running_loss = 0.0

    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for step, batch in enumerate(progress):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.autocast(
            device_type="cuda", dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model(**batch)
            loss = outputs.loss / ACCUMULATION_STEPS

        loss.backward()
        running_loss += loss.item() * ACCUMULATION_STEPS

        if (step + 1) % ACCUMULATION_STEPS == 0 or step + 1 == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        progress.set_postfix(loss=f"{running_loss/(step+1):.3f}")

    # Validation
    model.eval()
    predictions, targets = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            labels = batch["labels"].numpy()
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(
                device_type="cuda", dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                logits = model(**batch).logits

            predictions.extend(logits.argmax(dim=-1).cpu().tolist())
            targets.extend(labels.tolist())

    val_acc = accuracy_score(targets, predictions)
    val_f1 = f1_score(targets, predictions, average="weighted")
    print(
        f"Epoch {epoch+1}: "
        f"train_loss={running_loss/len(train_loader):.4f}, "
        f"val_accuracy={val_acc:.4f}, val_weighted_F1={val_f1:.4f}"
    )

    # Save each epoch so progress survives a runtime reset
    epoch_dir = os.path.join(checkpoint_dir, f"epoch_{epoch+1}")
    model.save_pretrained(epoch_dir)
    extractor.save_pretrained(epoch_dir)
    torch.save(
        {"epoch": epoch+1, "optimizer": optimizer.state_dict(),
         "val_f1": val_f1},
        os.path.join(epoch_dir, "training_state.pt")
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_pretrained(os.path.join(checkpoint_dir, "best"))
        extractor.save_pretrained(os.path.join(checkpoint_dir, "best"))
        print("Saved new best model.")

    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("Training complete. Best validation weighted F1:", best_f1)

Device: cuda
GPU: Tesla T4
Loaded train examples: 7109
Loaded validation examples: 1769


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  380MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  380MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

... (14 outputs left out)

Validation:   0%|          | 0/885 [00:00<?, ?it/s]

Epoch 3: train_loss=0.8475, val_accuracy=0.6478, val_weighted_F1=0.6461


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
import os

best_dir = "/content/drive/MyDrive/SpeechEmotionProject/combined_model_v1/best"
epoch3_dir = "/content/drive/MyDrive/SpeechEmotionProject/combined_model_v1/epoch_3"

for folder in [best_dir, epoch3_dir]:
    print("\n", folder)
    print("Exists:", os.path.isdir(folder))
    if os.path.isdir(folder):
        print("Files:", os.listdir(folder))


 /content/drive/MyDrive/SpeechEmotionProject/combined_model_v1/best
Exists: True
Files: ['config.json', 'model.safetensors', 'preprocessor_config.json']

 /content/drive/MyDrive/SpeechEmotionProject/combined_model_v1/epoch_3
Exists: True
Files: ['config.json', 'model.safetensors', 'preprocessor_config.json', 'training_state.pt']


In [5]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForAudioClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report

BASE = "/content/drive/MyDrive/SpeechEmotionProject"
MODEL_DIR = f"{BASE}/combined_model_v1/best"
TEST_DIR = f"{BASE}/processed_batches/test"

labels = [
    "angry", "calm", "disgust", "fearful",
    "happy", "neutral", "sad", "surprised"
]

class BatchAudioDataset(Dataset):
    def __init__(self, folder):
        self.files = sorted(
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.endswith(".npz")
        )
        self.index = []

        for file_id, path in enumerate(self.files):
            with np.load(path, allow_pickle=True) as data:
                self.index.extend(
                    (file_id, i) for i in range(len(data["labels"]))
                )

        self.cache_file = None
        self.cache_data = None

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        file_id, row_id = self.index[idx]
        path = self.files[file_id]

        if self.cache_file != path:
            if self.cache_data is not None:
                self.cache_data.close()
            self.cache_data = np.load(path, allow_pickle=True)
            self.cache_file = path

        audio = np.asarray(
            self.cache_data["input_values"][row_id],
            dtype=np.float32
        ).copy()
        label = int(self.cache_data["labels"][row_id])

        return {
            "input_values": torch.from_numpy(audio),
            "labels": label
        }

def collate_batch(batch):
    max_len = max(len(x["input_values"]) for x in batch)
    inputs = torch.zeros(len(batch), max_len)
    attention_mask = torch.zeros(len(batch), max_len, dtype=torch.long)

    for i, item in enumerate(batch):
        audio = item["input_values"]
        inputs[i, :len(audio)] = audio
        attention_mask[i, :len(audio)] = 1

    return {
        "input_values": inputs,
        "attention_mask": attention_mask,
        "labels": torch.tensor([x["labels"] for x in batch])
    }

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = AutoModelForAudioClassification.from_pretrained(
    MODEL_DIR
).to(device)
model.eval()

test_data = BatchAudioDataset(TEST_DIR)
test_loader = DataLoader(
    test_data,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_batch
)

y_true, y_pred = [], []

with torch.no_grad():
    for batch in test_loader:
        inputs = {
            k: batch[k].to(device)
            for k in ("input_values", "attention_mask")
        }
        logits = model(**inputs).logits
        predictions = logits.argmax(dim=-1).cpu().tolist()

        y_pred.extend(predictions)
        y_true.extend(batch["labels"].tolist())

print("\nTest samples:", len(y_true))
print("Test accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Test weighted F1:", round(f1_score(
    y_true, y_pred, average="weighted", zero_division=0
), 4))

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(8)),
    target_names=labels,
    zero_division=0
))

Device: cuda


Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]


Test samples: 2400
Test accuracy: 0.4813
Test weighted F1: 0.4797

Classification report:
              precision    recall  f1-score   support

       angry       0.92      0.81      0.86       400
        calm       0.00      0.00      0.00         0
     disgust       0.41      0.99      0.58       400
     fearful       1.00      0.16      0.27       400
       happy       0.37      0.41      0.39       400
     neutral       0.92      0.23      0.36       400
         sad       0.71      0.29      0.41       400
   surprised       0.00      0.00      0.00         0

    accuracy                           0.48      2400
   macro avg       0.54      0.36      0.36      2400
weighted avg       0.72      0.48      0.48      2400



In [6]:
import sounddevice as sd
import numpy as np
import torch
from IPython.display import Audio, display
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

MODEL_DIR = "/content/drive/MyDrive/SpeechEmotionProject/combined_model_v1/best"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForAudioClassification.from_pretrained(MODEL_DIR).to(device)
extractor = AutoFeatureExtractor.from_pretrained(MODEL_DIR)
model.eval()

print("Model ready!")

ModuleNotFoundError: No module named 'sounddevice'

In [7]:
import os

base = "/content/drive/MyDrive/SpeechEmotionProject"

for root, dirs, files in os.walk(base):
    dirs[:] = [d for d in dirs if d not in [
        "processed_batches", "data", "AudioWAV"
    ]]
    if "config.json" in files and (
        "model.safetensors" in files or "pytorch_model.bin" in files
    ):
        print(root)

/content/drive/MyDrive/SpeechEmotionProject/checkpoints/checkpoint-572
/content/drive/MyDrive/SpeechEmotionProject/checkpoints/checkpoint-715
/content/drive/MyDrive/SpeechEmotionProject/final_model
/content/drive/MyDrive/SpeechEmotionProject/experiment_2/checkpoint-143
/content/drive/MyDrive/SpeechEmotionProject/experiment_2/checkpoint-286
/content/drive/MyDrive/SpeechEmotionProject/experiment_2/checkpoint-429
/content/drive/MyDrive/SpeechEmotionProject/final_model_v2
/content/drive/MyDrive/SpeechEmotionProject/combined_model_v1/epoch_1
/content/drive/MyDrive/SpeechEmotionProject/combined_model_v1/best
/content/drive/MyDrive/SpeechEmotionProject/combined_model_v1/epoch_2
/content/drive/MyDrive/SpeechEmotionProject/combined_model_v1/epoch_3


In [8]:
import os
import json

base = "/content/drive/MyDrive/SpeechEmotionProject"

model_dirs = [
    "checkpoints/checkpoint-572",
    "checkpoints/checkpoint-715",
    "final_model",
    "experiment_2/checkpoint-143",
    "experiment_2/checkpoint-286",
    "experiment_2/checkpoint-429",
    "final_model_v2",
    "combined_model_v1/epoch_1",
    "combined_model_v1/epoch_2",
    "combined_model_v1/epoch_3",
    "combined_model_v1/best",
]

for name in model_dirs:
    path = os.path.join(base, name)
    config_path = os.path.join(path, "config.json")

    print(f"\n--- {name} ---")

    try:
        with open(config_path) as f:
            config = json.load(f)

        print("Architecture:", config.get("model_type"))
        print("Number of labels:", config.get("num_labels"))
        print("Labels:", config.get("id2label"))
        print("Model files:", [
            x for x in os.listdir(path)
            if x.endswith((".safetensors", ".bin"))
        ])
    except Exception as e:
        print("Could not read:", e)


--- checkpoints/checkpoint-572 ---
Architecture: wav2vec2
Number of labels: None
Labels: {'0': 'angry', '1': 'calm', '2': 'disgust', '3': 'fearful', '4': 'happy', '5': 'neutral', '6': 'sad', '7': 'surprised'}
Model files: ['model.safetensors', 'training_args.bin']

--- checkpoints/checkpoint-715 ---
Architecture: wav2vec2
Number of labels: None
Labels: {'0': 'angry', '1': 'calm', '2': 'disgust', '3': 'fearful', '4': 'happy', '5': 'neutral', '6': 'sad', '7': 'surprised'}
Model files: ['model.safetensors', 'training_args.bin']

--- final_model ---
Architecture: wav2vec2
Number of labels: None
Labels: {'0': 'angry', '1': 'calm', '2': 'disgust', '3': 'fearful', '4': 'happy', '5': 'neutral', '6': 'sad', '7': 'surprised'}
Model files: ['model.safetensors', 'training_args.bin']

--- experiment_2/checkpoint-143 ---
Architecture: wav2vec2
Number of labels: None
Labels: {'0': 'angry', '1': 'calm', '2': 'disgust', '3': 'fearful', '4': 'happy', '5': 'neutral', '6': 'sad', '7': 'surprised'}
Model 

In [9]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForAudioClassification
from sklearn.metrics import accuracy_score, f1_score

BASE = "/content/drive/MyDrive/SpeechEmotionProject"
TEST_DIR = f"{BASE}/processed_batches/test"

model_dirs = {
    "Previous checkpoint 572": f"{BASE}/checkpoints/checkpoint-572",
    "Previous checkpoint 715": f"{BASE}/checkpoints/checkpoint-715",
    "Previous final model": f"{BASE}/final_model",
    "Experiment 2 - 143": f"{BASE}/experiment_2/checkpoint-143",
    "Experiment 2 - 286": f"{BASE}/experiment_2/checkpoint-286",
    "Experiment 2 - 429": f"{BASE}/experiment_2/checkpoint-429",
    "Previous final model v2": f"{BASE}/final_model_v2",
    "Combined epoch 1": f"{BASE}/combined_model_v1/epoch_1",
    "Combined epoch 2": f"{BASE}/combined_model_v1/epoch_2",
    "Combined epoch 3": f"{BASE}/combined_model_v1/epoch_3",
    "Combined best": f"{BASE}/combined_model_v1/best",
}

class TestDataset(Dataset):
    def __init__(self, folder):
        self.files = sorted(
            os.path.join(folder, f) for f in os.listdir(folder)
            if f.endswith(".npz")
        )
        self.index = []
        for fi, path in enumerate(self.files):
            with np.load(path, allow_pickle=True) as d:
                self.index.extend((fi, i) for i in range(len(d["labels"])))
        self.cached_path = None
        self.cached_data = None

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        fi, ri = self.index[idx]
        path = self.files[fi]
        if path != self.cached_path:
            if self.cached_data is not None:
                self.cached_data.close()
            self.cached_data = np.load(path, allow_pickle=True)
            self.cached_path = path
        audio = np.asarray(
            self.cached_data["input_values"][ri], dtype=np.float32
        ).copy()
        return torch.from_numpy(audio), int(self.cached_data["labels"][ri])

def collate_fn(batch):
    max_len = max(len(x[0]) for x in batch)
    inputs = torch.zeros(len(batch), max_len)
    mask = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, (audio, _) in enumerate(batch):
        inputs[i, :len(audio)] = audio
        mask[i, :len(audio)] = 1
    labels = torch.tensor([y for _, y in batch])
    return inputs, mask, labels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset = TestDataset(TEST_DIR)
loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

# Gather true labels once, since every model uses the same test set.
y_true = []
for _, _, labels in loader:
    y_true.extend(labels.tolist())

results = []

for name, path in model_dirs.items():
    print(f"\nEvaluating: {name}")
    try:
        model = AutoModelForAudioClassification.from_pretrained(path).to(device)
        model.eval()
        predictions = []

        with torch.no_grad():
            for inputs, mask, _ in loader:
                logits = model(
                    input_values=inputs.to(device),
                    attention_mask=mask.to(device)
                ).logits
                predictions.extend(logits.argmax(dim=-1).cpu().tolist())

        results.append({
            "Model": name,
            "Accuracy (%)": round(100 * accuracy_score(y_true, predictions), 2),
            "Weighted F1": round(
                f1_score(y_true, predictions, average="weighted", zero_division=0), 4
            ),
            "Status": "OK"
        })
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        results.append({
            "Model": name, "Accuracy (%)": None,
            "Weighted F1": None, "Status": str(e)[:180]
        })
        print("Could not evaluate:", e)

results_df = pd.DataFrame(results)
display(results_df)

output_path = f"{BASE}/tess_model_comparison.csv"
results_df.to_csv(output_path, index=False)
print("Saved comparison to:", output_path)


Evaluating: Previous checkpoint 572


Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]


Evaluating: Previous checkpoint 715


Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]


Evaluating: Previous final model


Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

... (14 outputs left out)


Evaluating: Combined best


Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

,Model,Accuracy (%),Weighted F1,Status
0,Previous checkpoint 572,34.25,0.2851,OK
1,Previous checkpoint 715,34.62,0.2855,OK
2,Previous final model,34.25,0.2851,OK
3,Experiment 2 - 143,34.46,0.2807,OK
4,Experiment 2 - 286,34.46,0.2816,OK
5,Experiment 2 - 429,34.33,0.2803,OK
6,Previous final model v2,34.33,0.2803,OK
7,Combined epoch 1,40.12,0.3565,OK
8,Combined epoch 2,52.54,0.5267,OK
9,Combined epoch 3,48.12,0.4797,OK


Saved comparison to: /content/drive/MyDrive/SpeechEmotionProject/tess_model_comparison.csv


### Improvements Achieved

The speech-emotion recognition system was improved by fine-tuning a Wav2Vec2-based audio classification model using a combined dataset of CREMA-D, RAVDESS, and TESS. The resulting dataset contained 11,278 unique recordings across eight emotion categories.

The best-performing combined-model checkpoint achieved **52.54% accuracy and a weighted F1-score of 0.5267** on 2,400 TESS test recordings. Compared with the previous best checkpoint, which achieved 34.62% accuracy and a weighted F1-score of 0.2855 on the same test set, this represents an accuracy improvement of 17.92 percentage points.

The project also established a repeatable evaluation workflow, saved trained model checkpoints, and produced a comparative evaluation of 11 checkpoints. Real-time integration and performance on natural, spontaneous speech remain to be validated.


In [10]:
import os
import shutil

BASE = "/content/drive/MyDrive/SpeechEmotionProject"

# Keep the best-performing model from our TESS comparison.
MODEL = os.path.join(BASE, "combined_model_v1", "epoch_2")

# Create a dedicated backup folder.
BACKUP = os.path.join(BASE, "project_backup")
os.makedirs(BACKUP, exist_ok=True)

# Copy the selected model checkpoint.
shutil.copytree(
    MODEL,
    os.path.join(BACKUP, "best_tess_checkpoint"),
    dirs_exist_ok=True
)

# Copy the comparison results.
comparison = os.path.join(BASE, "tess_model_comparison.csv")
if os.path.exists(comparison):
    shutil.copy2(comparison, BACKUP)

# Record the measured results.
with open(os.path.join(BACKUP, "results.txt"), "w") as f:
    f.write("Speech Emotion Recognition Project\n")
    f.write("Best TESS checkpoint: combined_model_v1/epoch_2\n")
    f.write("TESS test samples: 2400\n")
    f.write("TESS test accuracy: 52.54%\n")
    f.write("TESS weighted F1: 0.5267\n")
    f.write("Validation results for epoch 2: pending evaluation\n")

# Create the ZIP backup in Google Drive.
zip_path = shutil.make_archive(
    os.path.join(BASE, "project_backup"),
    "zip",
    root_dir=BACKUP
)

print("Backup created successfully!")
print("ZIP location:", zip_path)
print("Backup contents:", os.listdir(BACKUP))

Backup created successfully!
ZIP location: /content/drive/MyDrive/SpeechEmotionProject/project_backup.zip
Backup contents: ['best_tess_checkpoint', 'tess_model_comparison.csv', 'results.txt']


In [11]:
import os
import numpy as np
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForAudioClassification
from sklearn.metrics import accuracy_score, f1_score

BASE = "/content/drive/MyDrive/SpeechEmotionProject"
TEST_DIR = f"{BASE}/processed_batches/test"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class TestDataset(Dataset):
    def __init__(self, folder):
        self.files = sorted(
            os.path.join(folder, f) for f in os.listdir(folder)
            if f.endswith(".npz")
        )
        self.index = []
        for fi, path in enumerate(self.files):
            with np.load(path, allow_pickle=True) as d:
                self.index.extend((fi, i) for i in range(len(d["labels"])))
        self.cache_path, self.cache = None, None

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        fi, ri = self.index[idx]
        path = self.files[fi]
        if path != self.cache_path:
            if self.cache is not None:
                self.cache.close()
            self.cache = np.load(path, allow_pickle=True)
            self.cache_path = path
        x = np.asarray(self.cache["input_values"][ri], dtype=np.float32).copy()
        return torch.from_numpy(x), int(self.cache["labels"][ri])

def collate_fn(batch):
    length = max(len(x) for x, _ in batch)
    inputs = torch.zeros(len(batch), length)
    mask = torch.zeros(len(batch), length, dtype=torch.long)
    for i, (x, _) in enumerate(batch):
        inputs[i, :len(x)] = x
        mask[i, :len(x)] = 1
    return inputs, mask, torch.tensor([y for _, y in batch])

dataset = TestDataset(TEST_DIR)
loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

y_true = []
for _, _, y in loader:
    y_true.extend(y.tolist())

models = {
    "Original pretrained": "facebook/wav2vec2-base",
    "Combined Epoch 2": f"{BASE}/combined_model_v1/epoch_2",
}

results = []

for name, model_path in models.items():
    print(f"\nEvaluating {name}...")
    try:
        if name == "Original pretrained":
            # Load the base Wav2Vec2 encoder with a newly initialized
            # classification head. This is NOT the original emotion model.
            model = AutoModelForAudioClassification.from_pretrained(
                model_path,
                num_labels=8,
                ignore_mismatched_sizes=True
            ).to(device)
        else:
            model = AutoModelForAudioClassification.from_pretrained(
                model_path
            ).to(device)

        model.eval()
        preds = []

        with torch.no_grad():
            for inputs, mask, _ in loader:
                logits = model(
                    input_values=inputs.to(device),
                    attention_mask=mask.to(device)
                ).logits
                preds.extend(logits.argmax(-1).cpu().tolist())

        results.append({
            "Model": name,
            "Accuracy (%)": round(100 * accuracy_score(y_true, preds), 2),
            "Weighted F1": round(
                f1_score(y_true, preds, average="weighted", zero_division=0), 4
            )
        })
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print("Error:", repr(e))

display(pd.DataFrame(results))


Evaluating Original pretrained...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_hid.weight           | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
projector.weight             | MISSING    | 
classifier.weight            | MISSING    | 
projector.bias               | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Evaluating Combined Epoch 2...


Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

,Model,Accuracy (%),Weighted F1
0,Original pretrained,11.58,0.0602
1,Combined Epoch 2,52.54,0.5267


In [12]:
from transformers import pipeline

original = pipeline(
    "audio-classification",
    model="Dpngtm/wav2vec2-emotion-recognition"
)

print("Original model labels:")
print(original.model.config.id2label)

config.json:   0%|          | 0.00/2.37k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

Original model labels:
{0: 'angry', 1: 'calm', 2: 'disgust', 3: 'fearful', 4: 'happy', 5: 'sad', 6: 'surprised'}


In [13]:
import os
from glob import glob

tess_path = "/content/tess_extracted"

print("TESS folder exists:", os.path.isdir(tess_path))

if os.path.isdir(tess_path):
    wav_files = glob(tess_path + "/**/*.wav", recursive=True)
    print("WAV files found:", len(wav_files))
    print("Example files:", wav_files[:3])

TESS folder exists: True
WAV files found: 2800
Example files: ['/content/tess_extracted/YAF_lose_angry.wav', '/content/tess_extracted/YAF_pad_ps.wav', '/content/tess_extracted/YAF_should_disgust.wav']


In [14]:
import os
import glob
import hashlib
import numpy as np
import librosa
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

# Original pretrained model is already loaded as `original`
audio_files = sorted(glob.glob(
    "/content/tess_extracted/**/*.wav", recursive=True
))

# Keep unique audio files using file contents
unique_files = {}
for path in audio_files:
    with open(path, "rb") as f:
        digest = hashlib.md5(f.read()).hexdigest()
    if digest not in unique_files:
        unique_files[digest] = path

test_files = sorted(unique_files.values())
print("Unique test recordings:", len(test_files))

# TESS filename emotion codes -> our common labels
emotion_map = {
    "angry": "angry",
    "disgust": "disgust",
    "fear": "fearful",
    "happy": "happy",
    "neutral": "neutral",
    "ps": "surprised",  # pleasant surprise
    "sad": "sad",
}

y_true, y_pred = [], []
skipped = []

for path in tqdm(test_files):
    filename = os.path.basename(path).lower()
    emotion_code = filename.rsplit("_", 1)[-1].replace(".wav", "")

    if emotion_code not in emotion_map:
        skipped.append(filename)
        continue

    # Load original waveform at 16 kHz
    audio, sr = librosa.load(path, sr=16000, mono=True)

    # Get original model prediction
    result = original(
        {"raw": audio, "sampling_rate": 16000}
    )
    predicted = result[0]["label"].lower()

    # Normalize possible label variants
    aliases = {
        "anger": "angry",
        "fear": "fearful",
        "happiness": "happy",
        "sadness": "sad",
        "surprise": "surprised",
    }
    predicted = aliases.get(predicted, predicted)

    # Exclude classes not represented by the original model
    if predicted not in set(emotion_map.values()):
        skipped.append(filename)
        continue

    y_true.append(emotion_map[emotion_code])
    y_pred.append(predicted)

print("\nEvaluated:", len(y_true))
print("Skipped:", len(skipped))
print(f"Accuracy: {accuracy_score(y_true, y_pred) * 100:.2f}%")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted'):.4f}")
print("\nClassification report:")
print(classification_report(y_true, y_pred, zero_division=0))

Unique test recordings: 2800


  0%|          | 0/2800 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Evaluated: 2400
Skipped: 400
Accuracy: 99.88%
Weighted F1: 0.9987

Classification report:
              precision    recall  f1-score   support

       angry       1.00      1.00      1.00       400
     disgust       1.00      0.99      1.00       400
     fearful       1.00      1.00      1.00       400
       happy       1.00      1.00      1.00       400
         sad       1.00      1.00      1.00       400
   surprised       0.99      1.00      1.00       400

    accuracy                           1.00      2400
   macro avg       1.00      1.00      1.00      2400
weighted avg       1.00      1.00      1.00      2400



In [15]:
print("Model:", original.model.name_or_path)
print("Labels:", original.model.config.id2label)
print("Sampling rate:", original.feature_extractor.sampling_rate)

Model: Dpngtm/wav2vec2-emotion-recognition
Labels: {0: 'angry', 1: 'calm', 2: 'disgust', 3: 'fearful', 4: 'happy', 5: 'sad', 6: 'surprised'}
Sampling rate: 16000


In [16]:
import os
import glob
import numpy as np

base = "/content/drive/MyDrive/SpeechEmotionProject"

print("Processed dataset folders:")
for split in ["train", "validation", "test"]:
    folder = os.path.join(base, "processed_batches", split)
    files = glob.glob(os.path.join(folder, "*.npz"))
    print(f"{split}: {len(files)} batch files")

Processed dataset folders:
train: 112 batch files
validation: 28 batch files
test: 38 batch files


In [17]:
import os
import glob
import numpy as np

base = "/content/drive/MyDrive/SpeechEmotionProject/processed_batches"

for split in ["train", "validation", "test"]:
    files = sorted(glob.glob(f"{base}/{split}/*.npz"))
    print(f"\n--- {split.upper()} ---")

    if not files:
        print("No batch files found")
        continue

    with np.load(files[0], allow_pickle=True) as data:
        print("File:", os.path.basename(files[0]))
        print("Arrays:", data.files)

        for key in data.files:
            arr = data[key]
            print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

            if arr.dtype == object and arr.size:
                print("  Example metadata:", arr.flat[0])


--- TRAIN ---
File: batch_0000.npz
Arrays: ['input_values', 'labels']
input_values: shape=(64,), dtype=object
  Example metadata: [ 0.0776542   0.06427182  0.04845627 ...  0.00709256 -0.00020692
  0.00709256]
labels: shape=(64,), dtype=int64

--- VALIDATION ---
File: batch_0000.npz
Arrays: ['input_values', 'labels']
input_values: shape=(64,), dtype=object
  Example metadata: [0.15529047 0.16454919 0.19510296 ... 0.00066983 0.00066983 0.00066983]
labels: shape=(64,), dtype=int64

--- TEST ---
File: batch_0000.npz
Arrays: ['input_values', 'labels']
input_values: shape=(64,), dtype=object
  Example metadata: [ 0.00037356 -0.01585048 -0.01691334 ...  0.00450197  0.00481622
  0.00073519]
labels: shape=(64,), dtype=int64


In [18]:
import os

base = "/content/drive/MyDrive/SpeechEmotionProject"

for root, dirs, files in os.walk(base):
    for name in files:
        lower = name.lower()
        if any(word in lower for word in [
            "split", "manifest", "metadata", "index",
            "dataset", "prepare", "preprocess", "train"
        ]) and lower.endswith((".py", ".csv", ".json", ".txt", ".pkl")):
            print(os.path.join(root, name))

/content/drive/MyDrive/SpeechEmotionProject/checkpoints/checkpoint-572/trainer_state.json
/content/drive/MyDrive/SpeechEmotionProject/checkpoints/checkpoint-715/trainer_state.json
/content/drive/MyDrive/SpeechEmotionProject/final_model/preprocessor_config.json
/content/drive/MyDrive/SpeechEmotionProject/experiment_2/checkpoint-143/trainer_state.json
/content/drive/MyDrive/SpeechEmotionProject/experiment_2/checkpoint-286/trainer_state.json
/content/drive/MyDrive/SpeechEmotionProject/experiment_2/checkpoint-429/trainer_state.json
/content/drive/MyDrive/SpeechEmotionProject/final_model_v2/preprocessor_config.json
/content/drive/MyDrive/SpeechEmotionProject/processed_batches/index.csv
/content/drive/MyDrive/SpeechEmotionProject/processed_batches/train.csv
/content/drive/MyDrive/SpeechEmotionProject/combined_model_v1/epoch_1/preprocessor_config.json
/content/drive/MyDrive/SpeechEmotionProject/combined_model_v1/best/preprocessor_config.json
/content/drive/MyDrive/SpeechEmotionProject/combine

In [19]:
import pandas as pd
import os

base = "/content/drive/MyDrive/SpeechEmotionProject/processed_batches"

for name in ["index.csv", "train.csv"]:
    path = os.path.join(base, name)

    print(f"\n{'=' * 15} {name} {'=' * 15}")

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(5).to_string(index=False))

    for col in df.columns:
        if any(word in col.lower() for word in
               ["dataset", "source", "split", "emotion"]):
            print(f"\nValues in {col}:")
            print(df[col].value_counts(dropna=False).to_string())


=============== index.csv ===============
Shape: (11278, 5)
Columns: ['path', 'emotion', 'label', 'dataset', 'speaker']
                                                 path emotion  label dataset    speaker
/content/crema_real/data/AudioWAV/1029_MTI_NEU_XX.wav neutral      5 CREMA-D CREMA_1029
/content/crema_real/data/AudioWAV/1011_IOM_DIS_XX.wav disgust      2 CREMA-D CREMA_1011
/content/crema_real/data/AudioWAV/1070_TIE_SAD_XX.wav     sad      6 CREMA-D CREMA_1070
/content/crema_real/data/AudioWAV/1008_IWL_ANG_XX.wav   angry      0 CREMA-D CREMA_1008
/content/crema_real/data/AudioWAV/1034_TSI_HAP_XX.wav   happy      4 CREMA-D CREMA_1034

Values in emotion:
emotion
disgust      1863
angry        1863
sad          1862
fearful      1862
happy        1861
neutral      1583
surprised     192
calm          192

Values in dataset:
dataset
CREMA-D    7439
TESS       2400
RAVDESS    1439

=============== train.csv ===============
Shape: (7109, 5)
Columns: ['path', 'emotion', 'label', 'data

In [20]:
import os
import pandas as pd

base = "/content/drive/MyDrive/SpeechEmotionProject/processed_batches"

for filename in ["train.csv", "validation.csv", "test.csv"]:
    path = os.path.join(base, filename)

    print(f"\n--- {filename} ---")

    if os.path.exists(path):
        df = pd.read_csv(path)
        print("Rows:", len(df))
        print(df["dataset"].value_counts().to_dict())
    else:
        print("File not found")


--- train.csv ---
Rows: 7109
{'CREMA-D': 5970, 'RAVDESS': 1139}

--- validation.csv ---
Rows: 1769
{'CREMA-D': 1469, 'RAVDESS': 300}

--- test.csv ---
Rows: 2400
{'TESS': 2400}


In [21]:
import os

checkpoint = (
    "/content/drive/MyDrive/SpeechEmotionProject/"
    "combined_model_v1/epoch_2"
)

print("Checkpoint exists:", os.path.isdir(checkpoint))
print("Files:", sorted(os.listdir(checkpoint)) if os.path.isdir(checkpoint) else [])

Checkpoint exists: True
Files: ['config.json', 'model.safetensors', 'preprocessor_config.json', 'training_state.pt']


In [22]:
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
import os

checkpoint = (
    "/content/drive/MyDrive/SpeechEmotionProject/"
    "combined_model_v1/epoch_2"
)

feature_extractor = AutoFeatureExtractor.from_pretrained(checkpoint)
model = AutoModelForAudioClassification.from_pretrained(checkpoint)

print("Model loaded successfully!")
print("Emotion labels:", model.config.id2label)
print("Sampling rate:", feature_extractor.sampling_rate)

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

Model loaded successfully!
Emotion labels: {0: 'angry', 1: 'calm', 2: 'disgust', 3: 'fearful', 4: 'happy', 5: 'neutral', 6: 'sad', 7: 'surprised'}
Sampling rate: 16000
